# অলীকবচন — LLM Hallucination Detection

**Competition notebook** for the Bengali LLM hallucination detection task. The pipeline classifies whether a model's Bengali response to a question is *faithful* (label `1`) or *hallucinated* (label `0`).

## Fix Summary

All 10 identified failure categories from the prior version are addressed here:

| # | Fix | Description | Errors Fixed |
|---|-----|-------------|-------------|
| 1 | Relation-span context verification | Extracts the exact date/event sentence from context rather than doing a naive substring search | 93 |
| 2 | Extended answer-type gate | Expanded from year-only to cover `where`, `who`, `age`, and `count` question types | 4+ |
| 3 | Strict short-answer comparison | Exact match for answers ≤ 3 tokens; Jaccard only for longer answers | ~40 |
| 4 | Relaxed QB threshold for language route | Lower similarity threshold (0.82) for Bengali grammar questions that are frequently paraphrased | 52 |
| 5 | Expanded math routing | Adds day-of-week, age, verbal percentages, compound interest, profit/loss, LCM/GCD | 10 |
| 6 | L2 disagree → LLM judge | Squad disagreements are re-routed to the LLM judge instead of being auto-labelled `0` | 27 |
| 7 | Relation-extraction helper | New `relation_span()` function used by Layer 3 to scope context verification | 93 |
| 8 | Reduced fallback prior population | Undecided rows go to LLM judge; prior is last resort only | all routes |
| A | Date/relation-aware judge prompt | Forces the judge to verify the *specific* event's date, not any date in the passage | 93 |
| B | Bengali linguistics specialist prompt | Dedicated prompt + answer-first mode for grammar/language questions | 63 |
| C | Math solver with sign/day/fraction support | Full step-by-step solver covering edge cases: signs, days-of-week, fractions | 14 |
| D | Closed-book factual judge | Instructs the judge that absence from the retrieval bank does not imply hallucination | 27 |

**Expected impact:** 210 audit-identified errors addressed → macro F1 gain.

**Environment:** GPU T4 ×2 · Internet ON · Run All


## Cell 1 — Environment Setup

Installs all required packages in the correct order (`vllm` first, then `--no-deps` extras) and prints a version report to confirm the environment is healthy before the pipeline runs.

**Key packages:** `vllm`, `bitsandbytes`, `accelerate`, `sentence-transformers`, `rank-bm25`, `sympy`


In [1]:
# Install order is critical: vllm first, then --no-deps extras.
import subprocess, sys, time
T_INSTALL = time.time()

def piprun(args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                       capture_output=True, text=True)
    tail = (r.stderr or "")[-300:]
    print(f"  pip {' '.join(args[:2])}... rc={r.returncode} {('| ' + tail) if tail.strip() else ''}")

piprun(["-U", "vllm"])
piprun(["-U", "bitsandbytes>=0.46.1", "accelerate", "protobuf", "rank-bm25", "sympy"])
piprun(["--no-deps", "sentence-transformers"])

print(f"\ninstalls done in {time.time()-T_INSTALL:.0f}s\n")

import importlib
print("="*60); print("VERSION REPORT"); print("="*60)
for mod in ("torch", "transformers", "vllm", "sentence_transformers",
            "datasets", "numpy", "sklearn", "rank_bm25", "sympy"):
    try:
        m = importlib.import_module(mod)
        print(f"  {mod:22s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"  {mod:22s} IMPORT FAILED: {type(e).__name__}: {str(e)[:60]}")
import torch
print(f"  cuda available        {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
print("="*60)


  pip -U vllm... rc=0 | but you have starlette 1.3.1 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.44.0 which is incompatible.
gradio 5.50.0 requires starlette<1.0,>=0.40.0, but you have starlette 1.3.1 which is incompatible.

  pip -U bitsandbytes>=0.46.1... rc=0 
  pip --no-deps sentence-transformers... rc=0 

installs done in 267s

VERSION REPORT
  torch                  2.11.0+cu130
  transformers           5.14.1
  vllm                   0.25.1
  sentence_transformers  IMPORT FAILED: RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FF
  datasets               5.0.0
  numpy                  2.0.2
  sklearn                1.6.1
  rank_bm25              ?
  sympy                  1.14.0
  cuda available        True | GPUs: 2


## Cell 2 — Exam-Bank Preload

Downloads and indexes four Bengali QA datasets that form the retrieval bank used by Layers 1 and 2.

| Dataset | Source |
|---------|--------|
| BEnQA | Bengali MCQ exam questions (Bengali-only columns extracted) |
| BanglaMedQA | Bengali medical QA |
| BanglaRQA | Bengali reading comprehension |
| BanglaQuAD | Bengali SQuAD-style QA |

All sources are deduplicated by normalised question prefix before being stored in `qb_q` / `qb_a`.


In [2]:
# Sources: BEnQA, BanglaMedQA, BanglaRQA, BanglaQuAD
import urllib.request, zipfile, json, re, unicodedata
from pathlib import Path
import pandas as pd
from datasets import load_dataset

WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)

qb_q, qb_a = [], []

def _norm_probe(t):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(t))).strip().lower()

# ---- BEnQA: Bengali columns only ----
try:
    zp = WORK / "benqa.zip"
    if not zp.exists() or zp.stat().st_size < 1000:
        urllib.request.urlretrieve(
            "https://github.com/sheikhshafayat/BEnQA/archive/refs/heads/master.zip", zp)
    zd = WORK / "benqa_x"
    if not zd.exists():
        with zipfile.ZipFile(zp) as z: z.extractall(zd)
    n0 = len(qb_q)
    for p in zd.rglob("*.csv"):
        try:
            d = pd.read_csv(p)
            d.columns = [str(c).strip().lower() for c in d.columns]
            qcol = "bengali question" if "bengali question" in d.columns else None
            bn_opts = [c for c in ("a bn", "b bn", "c bn", "d bn", "e bn") if c in d.columns]
            acol = next((c for c in ("answer", "answer bn", "correct answer",
                                     "correct_ans", "ans") if c in d.columns), None)
            if not (qcol and bn_opts and acol):
                print(f"  BEnQA {p.name}: SKIP, cols={list(d.columns)}"); continue
            LM = {"a":0,"b":1,"c":2,"d":3,"e":4}
            added = 0
            for _, r in d.iterrows():
                q = str(r[qcol]).strip()
                a = str(r[acol]).strip().lower()
                idx = LM.get(a[:1])
                if q and idx is not None and idx < len(bn_opts):
                    gold = str(r[bn_opts[idx]]).strip()
                    if gold and gold != "nan":
                        qb_q.append(q); qb_a.append(gold); added += 1
            print(f"  BEnQA {p.name}: +{added}")
        except Exception as e:
            print(f"  BEnQA {p.name}: {str(e)[:60]}")
    print(f"  BEnQA total: +{len(qb_q)-n0}")
except Exception as e:
    print(f"  BEnQA failed: {str(e)[:80]}")

# ---- BanglaMedQA ----
try:
    ds = load_dataset("ajwad-abrar/BanglaMedQA", split="train")
    n0 = len(qb_q)
    for r in ds:
        row = {str(k).lower(): v for k, v in r.items()}
        q = str(row.get("question", "")).strip()
        a = row.get("answer") or row.get("correct_answer") or ""
        if q and str(a).strip():
            qb_q.append(q); qb_a.append(str(a).strip())
    print(f"  BanglaMedQA: +{len(qb_q)-n0}")
except Exception as e:
    print(f"  BanglaMedQA failed: {str(e)[:80]}")

# ---- BanglaRQA ----
try:
    n0 = len(qb_q)
    for fn in ("Train.json", "Validation.json", "Test.json"):
        fp = WORK / f"brqa_{fn}"
        if not fp.exists():
            urllib.request.urlretrieve(
                f"https://huggingface.co/datasets/sartajekram/BanglaRQA/resolve/main/{fn}", fp)
        obj = json.load(open(fp, encoding="utf-8"))
        data = obj.get("data", obj) if isinstance(obj, dict) else obj
        added = 0
        for art in data:
            for qa in (art.get("qas") or art.get("questions") or []):
                q = qa.get("question_text") or qa.get("question") or ""
                ans = qa.get("answers") or {}
                texts = (ans.get("answer_text") or ans.get("text") or []) \
                        if isinstance(ans, dict) else ans
                a = str(texts[0]).strip() if texts else ""
                if str(qa.get("is_answerable", 1)) in ("0", "False", "no"):
                    continue
                if q and a:
                    qb_q.append(str(q).strip()); qb_a.append(a); added += 1
        print(f"  BanglaRQA {fn}: +{added}")
    print(f"  BanglaRQA total: +{len(qb_q)-n0}")
except Exception as e:
    print(f"  BanglaRQA failed: {str(e)[:80]}")

# ---- dedup ----
seen, Q2, A2 = set(), [], []
for q, a in zip(qb_q, qb_a):
    k = _norm_probe(q)[:80]
    if k not in seen:
        seen.add(k); Q2.append(q); A2.append(a)
qb_q, qb_a = Q2, A2
print(f"PRE-LOADED BANK: {len(qb_q)} unique QA pairs")

def _ingest_qa_json(obj, tag):
    n0 = len(qb_q)
    if isinstance(obj, dict) and "data" in obj:
        for art in obj["data"]:
            for para in art.get("paragraphs", []):
                for qa in para.get("qas", []):
                    ans = qa.get("answers", [])
                    if isinstance(ans, dict):
                        ans = ans.get("text", [])
                        ans = [{"text": t} for t in ans]
                    if not ans: continue
                    a0 = ans[0].get("text", "") if isinstance(ans[0], dict) else str(ans[0])
                    q0 = qa.get("question")
                    if q0 and a0:
                        qb_q.append(str(q0).strip()); qb_a.append(str(a0).strip())
    elif isinstance(obj, list):
        for r in obj:
            if not isinstance(r, dict): continue
            row = {str(k).lower(): v for k, v in r.items()}
            q0 = row.get("question") or row.get("question_text") or ""
            a0 = row.get("answer") or row.get("correct_answer") or ""
            if isinstance(a0, dict):
                a0 = (a0.get("text") or [""])
                a0 = a0[0] if isinstance(a0, list) and a0 else ""
            if str(q0).strip() and str(a0).strip():
                qb_q.append(str(q0).strip()); qb_a.append(str(a0).strip())
    print(f"  {tag}: +{len(qb_q)-n0} (total {len(qb_q)})")

# BanglaQuAD
try:
    zp = WORK / "banglaquad.zip"
    if not zp.exists() or zp.stat().st_size < 1000:
        for br in ("main", "master"):
            try:
                urllib.request.urlretrieve(
                    f"https://codeload.github.com/rashad101/BanglaQuAD-LREC-COLING-24/zip/refs/heads/{br}", zp)
                if zp.stat().st_size > 1000: break
            except Exception: pass
    zd = WORK / "banglaquad_x"
    if not zd.exists():
        with zipfile.ZipFile(zp) as z: z.extractall(zd)
    for p in zd.rglob("*.json"):
        try:
            _ingest_qa_json(json.load(open(p, encoding="utf-8")), f"BanglaQuAD {p.name}")
        except Exception as e:
            print(f"  BanglaQuAD {p.name}: {str(e)[:60]}")
except Exception as e:
    print(f"  BanglaQuAD failed: {str(e)[:80]}")

# final dedup
seen, Q2, A2 = set(), [], []
for q, a in zip(qb_q, qb_a):
    k = _norm_probe(q)[:80]
    if k not in seen:
        seen.add(k); Q2.append(q); A2.append(a)
qb_q, qb_a = Q2, A2
print(f"BANK AFTER MERGE: {len(qb_q)} unique QA pairs")


  BEnQA 12th-Biology-I-few_shot.csv: SKIP, cols=['unnamed: 0', 'question', 'question bn', 'a', 'b', 'c', 'd', 'correct answer', 'diagram', 'year and board', 'gt sanity', 'a bn', 'b bn', 'c bn', 'd bn', 'correct answer bn', 'map', 'diagram bn', 'year and board bn', 'similarity score', 'red flag', 'yellow flag', 'recheck', 'question_fixed', 'question_type', 'llm_unparsed', 'english_response', 'bengali_response']
  BEnQA 12th-Chemistry-I-few_shot.csv: SKIP, cols=['unnamed: 0', 'question', 'question bn', 'a', 'b', 'c', 'd', 'correct answer', 'diagram', 'year and board', 'gt sanity', 'a bn', 'b bn', 'c bn', 'd bn', 'correct answer bn', 'map', 'diagram bn', 'year and board bn', 'similarity score', 'red flag', 'yellow flag', 'recheck', 'question_fixed', 'question_type', 'llm_unparsed', 'english_response', 'bengali_response']
  BEnQA 10th-Biology-few_shot.csv: SKIP, cols=['unnamed: 0', 'question', 'question bn', 'a', 'b', 'c', 'd', 'correct answer', 'diagram', 'year and board', 'gt sanity', 'a

README.md: 0.00B [00:00, ?B/s]

BanglaMMedBench.csv: 0.00B [00:00, ?B/s]

bangla-med-qa.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2999 [00:00<?, ? examples/s]

  BanglaMedQA: +2994
  BanglaRQA Train.json: +9008
  BanglaRQA Validation.json: +1126
  BanglaRQA Test.json: +1124
  BanglaRQA total: +11258
PRE-LOADED BANK: 17777 unique QA pairs
BANK AFTER MERGE: 17777 unique QA pairs


## Cell 3 — Normalisers, Routing & Deterministic Layers (L1 / L2 / L3)

The core deterministic pipeline. Rows that can be decided with high confidence without the LLM are labelled here.

### Text Normalisation
- `bn_norm` — NFC normalise, convert Bengali digits to ASCII, strip punctuation, lowercase
- `bn_norm_numeric` — wraps `bn_norm` after converting written Bengali number words to digits (e.g. *আঠারশ' বত্রিশ* → `1832`)
- `numbers_of` — extract all numeric tokens preserving fraction structure and sign
- `resp_agree` — semantic agreement check: exact for short answers (≤ 3 tokens), Jaccard for longer ones, with explicit polarity and fraction-order checks

### Question Routing
Prompts are routed to one of four tracks before any scoring:

| Route | Trigger patterns |
|-------|-----------------|
| `math` | arithmetic keywords, word problems, day-of-week, interest, profit/loss |
| `language` | Bengali grammar keywords: antonym, idiom, sandhi, samasa, etc. |
| `date` | year/date question patterns |
| `factual` | everything else |

### Decision Layers

**Layer 1 — Sample answer-key match**
TF-IDF character n-gram similarity against the labelled sample set. At ≥ 0.90 similarity:
- Exact question+response pair → copy the known label
- Sample label is 1 → `resp_agree` with the gold response
- Sample label is 0 → flag if the test response matches the known-wrong answer

**Layer 2 — Squad-BN gold answers**
TF-IDF match against `squad_bn` (up to 150k QA pairs). Applies the answer-type gate before accepting any gold answer. Language-route rows use a relaxed threshold (0.82). Disagreements are re-routed to the LLM judge rather than being auto-labelled 0.

**Layer 3 — Track-A context verification (Relation-Span Fix)**
For rows with a supplied context passage:
1. Detect the relation asked (founding, birth, death, publication, etc.)
2. Extract only the sentences in the context that contain that relation trigger
3. Check whether the response appears in those relation-specific sentences

This prevents false positives where a year that appears elsewhere in the passage is incorrectly accepted as supporting a different event.


In [3]:
import json, re, sys, time, unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize

T0 = time.time()
def log(msg=""):
    print(f"[+{time.time()-T0:5.0f}s] {msg}")

pd.set_option("display.max_colwidth", 100)
WORK = Path("/kaggle/working")

CFG = dict(
    T1_SAMPLE_MATCH   = 0.90,
    T2_SQUAD_MATCH    = 0.88,
    T2_LANG_MATCH     = 0.82,   # FIX 4: relaxed threshold for language route
    A_OVERLAP_FAITH   = 0.60,
    SQUAD_MAX_ROWS    = 150_000,
    SHORT_ANSWER_TOKS = 3,      # FIX 3: threshold for exact vs Jaccard compare
)
print("CONFIG:", CFG)

# ── S0. LOAD DATA ────────────────────────────────────────────────
log("S0 — loading data")

CANDIDATE_ROOTS = [
    Path("/kaggle/input/datasets/jubayer2604"),
    Path("/kaggle/input/jubayer2604"),
    Path("/kaggle/input"),
]

def find_file(*names):
    for root in CANDIDATE_ROOTS:
        if not root.exists(): continue
        for name in names:
            hits = list(root.rglob(name))
            if hits: return hits[0]
    return None

SAMPLE_PATH = find_file("dataset samples (1).json", "dataset samples.json", "dataset_samples.json")
TEST_PATH   = find_file("test set (3).csv", "test set.csv", "test.csv")
assert SAMPLE_PATH and TEST_PATH, f"missing files: sample={SAMPLE_PATH} test={TEST_PATH}"
print(f"  sample: {SAMPLE_PATH}\n  test:   {TEST_PATH}")

with open(SAMPLE_PATH, encoding="utf-8") as f:
    sample_df = pd.DataFrame(json.load(f))
test_df = pd.read_csv(TEST_PATH)

_NULL_TOKENS = {"[NULL]", "nan", "None", "", "null", "NULL"}
def norm_ctx(x):
    s = str(x).strip()
    return "" if s in _NULL_TOKENS else s

for df in (sample_df, test_df):
    if "context" not in df.columns: df["context"] = ""
    df["ctx"] = df["context"].apply(norm_ctx)
    df["has_ctx"] = df["ctx"].ne("")
    df["prompt_bn"] = df["prompt_bn"].astype(str)
    df["response_bn"] = df["response_bn"].astype(str)
if "id" not in test_df.columns:
    test_df = test_df.reset_index().rename(columns={"index": "id"})

print(f"  sample {sample_df.shape} | test {test_df.shape} | "
      f"test Track-A {test_df.has_ctx.sum()} / Track-B {(~test_df.has_ctx).sum()}")

# ── S1. TEXT NORMALIZATION ───────────────────────────────────────
log("S1 — normalizers")

BN2ASCII = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

_BN_NUM_WORDS = {
    "শূন্য":0,"এক":1,"দুই":2,"দু":2,"তিন":3,"চার":4,"পাঁচ":5,"ছয়":6,"ছয":6,
    "সাত":7,"আট":8,"নয়":9,"নয":9,"দশ":10,"এগারো":11,"এগার":11,"বারো":12,
    "বার":12,"তেরো":13,"তের":13,"চৌদ্দ":14,"পনেরো":15,"পনের":15,"ষোল":16,
    "সতেরো":17,"সতের":17,"আঠারো":18,"আঠার":18,"উনিশ":19,"বিশ":20,"একুশ":21,
    "বাইশ":22,"তেইশ":23,"চব্বিশ":24,"পঁচিশ":25,"ছাব্বিশ":26,"সাতাশ":27,
    "আঠাশ":28,"ঊনত্রিশ":29,"উনত্রিশ":29,"ত্রিশ":30,"একত্রিশ":31,"বত্রিশ":32,
    "তেত্রিশ":33,"চৌত্রিশ":34,"পঁয়ত্রিশ":35,"ছত্রিশ":36,"সাঁইত্রিশ":37,
    "আটত্রিশ":38,"ঊনচল্লিশ":39,"উনচল্লিশ":39,"চল্লিশ":40,"একচল্লিশ":41,
    "বিয়াল্লিশ":42,"তেতাল্লিশ":43,"চুয়াল্লিশ":44,"পঁয়তাল্লিশ":45,
    "ছেচল্লিশ":46,"সাতচল্লিশ":47,"আটচল্লিশ":48,"ঊনপঞ্চাশ":49,"উনপঞ্চাশ":49,
    "পঞ্চাশ":50,"একান্ন":51,"বাহান্ন":52,"তিপ্পান্ন":53,"চুয়ান্ন":54,
    "পঞ্চান্ন":55,"ছাপ্পান্ন":56,"সাতান্ন":57,"আটান্ন":58,"ঊনষাট":59,
    "উনষাট":59,"ষাট":60,"একষট্টি":61,"বাষট্টি":62,"তেষট্টি":63,"চৌষট্টি":64,
    "পঁয়ষট্টি":65,"ছেষট্টি":66,"সাতষট্টি":67,"আটষট্টি":68,"ঊনসত্তর":69,
    "উনসত্তর":69,"সত্তর":70,"একাত্তর":71,"বাহাত্তর":72,"তিয়াত্তর":73,
    "চুয়াত্তর":74,"পঁচাত্তর":75,"ছিয়াত্তর":76,"সাতাত্তর":77,"আটাত্তর":78,
    "ঊনআশি":79,"উনআশি":79,"আশি":80,"একাশি":81,"বিরাশি":82,"তিরাশি":83,
    "চুরাশি":84,"পঁচাশি":85,"ছিয়াশি":86,"সাতাশি":87,"আটাশি":88,"ঊনানব্বই":89,
    "উননব্বই":89,"নব্বই":90,"একানব্বই":91,"বিরানব্বই":92,"তিরানব্বই":93,
    "চুরানব্বই":94,"পঁচানব্বই":95,"ছিয়ানব্বই":96,"সাতানব্বই":97,
    "আটানব্বই":98,"নিরানব্বই":99,
    "শত":100,"হাজার":1000,"লক্ষ":100000,"লাখ":100000,"কোটি":10000000,
}
_SHO_RE = re.compile(r"([\u0980-\u09FF]+?)(?:শো|শ)(?=\s|$|[^\u0980-\u09FF])")

def bn_words_to_digits(text: str) -> str:
    t = re.sub(r"['''`]", "", str(text))
    def _sho(m):
        base = _BN_NUM_WORDS.get(m.group(1))
        return str(base * 100) if base and base < 100 else m.group(0)
    t = _SHO_RE.sub(_sho, t)
    toks = t.split()
    out, acc, has_acc = [], 0, False
    def _flush():
        nonlocal acc, has_acc
        if has_acc:
            out.append(str(acc)); acc, has_acc = 0, False
    for w in toks:
        wc = w.strip("।,.;:!\"()")
        v = None
        if wc in _BN_NUM_WORDS: v = _BN_NUM_WORDS[wc]
        elif wc.isdigit():      v = int(wc)
        if v is None:
            _flush(); out.append(w); continue
        if not has_acc:
            acc, has_acc = v, True
        elif v in (100, 1000, 100000, 10000000):
            acc *= v
        elif v < 100 and acc >= 100 and acc % 100 == 0:
            acc += v
        else:
            _flush(); acc, has_acc = v, True
    _flush()
    return " ".join(out)

def bn_norm(text: str) -> str:
    t = unicodedata.normalize("NFC", str(text))
    t = t.translate(BN2ASCII)
    t = re.sub(r"\*\*|__|`", " ", t)
    t = re.sub(r'[!@#$%^&*()_+={}|;<>?,/~`]', ' ', t)
    t = re.sub(r'[\[\]\-\\]', ' ', t)
    t = re.sub('[\"\u201c\u201d\u2018\u2019]', ' ', t)
    t = re.sub('[\u0964\u2013\u2014]', ' ', t)
    t = re.sub(r"\s+", " ", t).strip().lower()
    return t

def bn_norm_numeric(text: str) -> str:
    return bn_norm(bn_words_to_digits(str(text)))

_NUM_RE = re.compile(
    r"(?<![\w\d])[+-]?\d+(?:\.\d+)?"
    r"(?:\s*/\s*[+-]?\d+(?:\.\d+)?)?"
)

def numbers_of(text: str) -> tuple:
    """
    Extract numbers in their original order while preserving fraction structure
    and an explicit sign. This keeps 1/2 distinct from 2/1 and +1/2 distinct
    from -1/2.
    """
    raw = unicodedata.normalize("NFC", bn_words_to_digits(str(text)))
    raw = raw.translate(BN2ASCII).lower()
    return tuple(re.sub(r"\s+", "", m.group(0)) for m in _NUM_RE.finditer(raw))

_POSITIVE_RE = re.compile(
    r"\b(?:positive|plus)\b|ধনাত্মক|ইতিবাচক|পজিটিভ",
    re.I | re.U,
)
_NEGATIVE_RE = re.compile(
    r"\b(?:negative|minus)\b|ঋণাত্মক|নেতিবাচক|নেগেটিভ",
    re.I | re.U,
)

def polarity_of(text: str):
    raw = unicodedata.normalize("NFC", str(text)).lower()
    has_pos = bool(_POSITIVE_RE.search(raw))
    has_neg = bool(_NEGATIVE_RE.search(raw))
    if has_pos and not has_neg:
        return 1
    if has_neg and not has_pos:
        return -1
    return None

_STOP = set("এর কে কি কী কোন কত হয় হলো ছিল করে থেকে একটি টি জন সালে সাল "
            "নাম বলা মোট প্রায় হচ্ছে এবং বা ও the a an of in is was".split())

def core_tokens(text: str) -> set:
    return {w for w in bn_norm_numeric(text).split()
            if w not in _STOP and len(w) > 1}

# ── FIX 3: STRICT SHORT-ANSWER COMPARISON ───────────────────────
def resp_agree(a: str, b: str) -> bool:
    """
    For short answers, require near exact agreement. Numeric equality alone is
    not enough when the answers contain conflicting polarity markers.
    """
    na, nb = bn_norm_numeric(a), bn_norm_numeric(b)
    if not na or not nb:
        return False

    # Check structured numeric content before normalized equality because bn_norm
    # intentionally removes signs and fraction punctuation.
    num_a, num_b = numbers_of(a), numbers_of(b)
    if num_a and num_b:
        if num_a != num_b:
            return False

        pol_a, pol_b = polarity_of(a), polarity_of(b)

        # An explicit negative marker cannot agree with a positive or unsigned answer.
        if (pol_a == -1 and pol_b != -1) or (pol_b == -1 and pol_a != -1):
            return False

        # Explicit opposite polarity words must never agree.
        if pol_a is not None and pol_b is not None and pol_a != pol_b:
            return False

        return True

    if na == nb:
        return True

    toks_a = na.split()
    toks_b = nb.split()
    short = min(len(toks_a), len(toks_b)) <= CFG["SHORT_ANSWER_TOKS"]
    if short:
        return (na in nb or nb in na) and (len(na) >= 2 and len(nb) >= 2)

    if len(na) >= 3 and len(nb) >= 3 and (na in nb or nb in na):
        return True

    ca, cb = core_tokens(a), core_tokens(b)
    if not ca or not cb:
        return False

    j = len(ca & cb) / len(ca | cb)
    return j >= 0.5

# self-tests
_tests = [
    ("১০০ : ১০০.৬", "১০০ : ১০০.৩", False),
    ("০.০৯২৯", "৬.৪৫", False),
    ("চুরুলিয়া, বর্ধমান", "টুঙ্গিপাড়া, ফরিদপুর", False),
    ("স্বাধীন বাংলাদেশের প্রথম চলচ্চিত্রটির নাম ছিল **সুকুমারী**।", "সুকুমারী", True),
    ("৭ বছর", "৬", False),
    ("আঠারশ' বত্রিশ সালে", "১৮৩২", True),
    ("মেহদী হাসান খান", "মেহদী হাসান খান", True),
    # FIX 3 new tests:
    ("বৃহস্পতিবার", "শুক্রবার", False),        # day names — must not match
    ("১৯১৩", "১৯৮৩", False),                   # years — must not match
    ("positive 1/2", "negative 1/2", False),    # polarity words matter
    ("+1/2", "-1/2", False),                       # explicit signs matter
    ("1/2", "2/1", False),                         # fraction order matters
    ("ধনাত্মক ১/২", "ঋণাত্মক ১/২", False),         # Bengali polarity matters
    ("স্থির", "চঞ্চল", False),                   # antonyms must not match
    ("স্থির", "স্থির", True),                    # exact same — must match
]
print("  resp_agree self-tests:")
all_ok = True
for a, b, want in _tests:
    got = resp_agree(a, b)
    ok = got == want
    all_ok = all_ok and ok
    print(f"    {'OK ' if ok else 'FAIL'} agree({a[:30]!r},{b[:30]!r}) -> {got} (want {want})")
assert all_ok, "resp_agree self-tests failed — abort"

# ── S2. ROUTING — FIX 5: MATH ROUTING EXPANDED ──────────────────
log("S2 — routing (math expanded)")

MATH_RE = re.compile(
    # Original patterns
    r"কত[টি]?\s*(হবে|হয়|হওয়ার)|যোগফল|বিয়োগফল|গুণফল|ভাগফল|সম্ভাব্যতা|"
    r"সম্ভাবনা|শতকরা|গাণিতিক|গুণিতক|মৌলিক সংখ্যা|ক্ষেত্রফল|পরিসীমা|"
    r"অঙ্কবিশিষ্ট|সমীকরণ|থেকে.*পর্যন্ত.*সংখ্যা|অনুপাত|লসাগু|গসাগু|"
    r"\d+\s*[\+\-\×\÷\*\/]|"
    # Word problem templates
    r"একটি মিশ্রণে|দুইটি শহরের মধ্যে দূরত্ব|একা একটি কাজ|"
    r"দুইজন সাইকেল আরোহী|একই স্থান থেকে একই|পণ্যের দাম প্রথম|"
    r"পণ্যের প্রাথমিক মূল্য|সংকেত বাতি যথাক্রমে|বাস স্টপেজ থেকে|"
    r"গড়\s*(কত|বয়স|মান)|সুদে|সুদের হার|আসল|লাভ.*ক্ষতি|ক্ষতি.*লাভ|"
    r"ক্রয়.*বিক্রয়|বিক্রয়.*ক্রয়|"
    # FIX 5 NEW: day-of-week, age, percent verbal, compound interest, ratio
    r"কোন বার\b|কোন দিন\b|বার হবে|দিন হবে|"        # day-of-week
    r"বয়স কত|কত বছর বয়সে|বছর বয়সে|বছর আগে|বছর পরে|"  # age calculations
    r"শতাংশ হ্রাস|শতাংশ বৃদ্ধি|শতাংশ কমলে|শতাংশ বাড়লে|"  # verbal percentage
    r"চক্রবৃদ্ধি সুদ|সরল সুদ|মূলধন|টাকায় সুদ|"    # interest
    r"ক্ষতি হয়|লাভ হয়|বিক্রয় মূল্য|ক্রয় মূল্য|"   # profit/loss verbal
    r"লগারিদম|বর্গমূল|ঘনমূল|বর্গ সংখ্যা|পূর্ণ বর্গ|"  # algebra
    r"সমান্তর ধারা|গুণোত্তর ধারা|ধারার সমষ্টি"      # series
)
LANG_RE = re.compile(
    r"অর্থ\s*কী|ভাবার্থ|সন্ধি|সমাস|বানান.*শুদ্ধ|শুদ্ধ.*বানান|"
    r"প্রতিশব্দ|বিপরীত\s*শব্দ|ব্যাকরণ|শব্দের\s*অর্থ|ইংরেজি.*অর্থ|"
    r"কারক|বিভক্তি|প্রত্যয়|উপসর্গ|পদ\s*কী|বাগধারা|প্রবাদ|ছন্দ|অলঙ্কার|ভাষা.*কি\s*করে|"
    r"বাগধারার অর্থ|শব্দের ভাবার্থ|সমাস নির্ণয়|সন্ধি বিচ্ছেদ|উপসর্গের শ্রেণি|"
    r"শব্দটির অর্থ|কোন শ্রেণির"
)
DATE_RE = re.compile(r"কত\s*সালে|কোন\s*সালে|কবে\s")

def route(p: str) -> str:
    if MATH_RE.search(p): return "math"
    if LANG_RE.search(p): return "language"
    if DATE_RE.search(p): return "date"
    return "factual"

test_df["route"] = test_df["prompt_bn"].apply(route)
sample_df["route"] = sample_df["prompt_bn"].apply(route)
print(f"  test routing: {test_df['route'].value_counts().to_dict()}")
print(f"  sample routing: {sample_df['route'].value_counts().to_dict()}")

prior = sample_df.groupby("route")["label"].mean()
ROUTE_PRIOR = {r: int(prior.get(r, 0.5) >= 0.5) for r in ("math","language","date","factual")}
print(f"  per-route majority priors: {ROUTE_PRIOR}")

# ── FIX 2: EXPANDED ANSWER TYPE GATE ────────────────────────────
_YEAR_Q   = re.compile(r"কত\s*সালে|কোন\s*সালে|(?:^|\s)কবে(?:\s|$|[?।,])|কোন\s*বছর")
_COUNT_Q  = re.compile(r"কত[টি](?:\s|$|[?।,])|কয়টি|কতগুলো|কত\s*জন|সংখ্যা\s*কত")
_WHO_Q    = re.compile(r"(?:^|\s)কে(?:\s|$|[?।,])|কার\s*নাম|কাকে|কে\s*ছিলেন|কে\s*প্রতিষ্ঠা")
_WHERE_Q  = re.compile(r"(?:^|\s)কোথায়(?:\s|$|[?।,])|কোন স্থানে|কোন জায়গায়|কোথা থেকে")
_AGE_Q    = re.compile(r"কত বছর বয়সে|বয়সে মৃত্যু|কত বছর বয়স")
_4DIGIT   = re.compile(r"\b\d{4}\b")
_ANYNUM   = re.compile(r"\d")
_BN_PLACE = re.compile(r"[\u0980-\u09FF]{3,}")   # 3+ Bengali letters = likely a name/place

def answer_type_ok(question: str, gold: str) -> bool:
    """
    FIX 2: Extended type gate. Only blocks clear mismatches; unknown passes.
    """
    q = str(question)
    g = bn_norm_numeric(str(gold))
    if _YEAR_Q.search(q):
        # Year question must have a 4-digit year in the response
        return bool(_4DIGIT.search(g))
    if _COUNT_Q.search(q):
        # Count question must have some digit
        return bool(_ANYNUM.search(g))
    if _WHO_Q.search(q):
        # Who question: response must not be purely a year/number
        g_stripped = g.replace(" ", "")
        return not (g_stripped.isdigit() and len(g_stripped) >= 4)
    if _WHERE_Q.search(q):
        # Where question: response must not be purely a year
        return not bool(_4DIGIT.match(g.strip()))
    if _AGE_Q.search(q):
        # Age question: response must contain a number (not a full date string)
        return bool(_ANYNUM.search(g)) and not bool(_4DIGIT.search(g))
    return True

# self-tests for type gate
_gate_tests = [
    ("তারেক মাসুদ পরিচালিত সর্বশেষ বাংলা চলচ্চিত্রটি কত সালে মুক্তি পায়?", "রানওয়ে", False),
    ("কত সালে সর্বপ্রথম ডায়নামো নির্মিত হয়েছিল ?", "১৮৩২", True),
    ("অভ্র কিবোর্ড কে উদ্ভাবন করেন ?", "মেহদী হাসান খান", True),
    ("বাংলাদেশের প্রথম রাষ্ট্রপতি কে ?", "১৯৭১", False),
    ("বাংলা ভাষা আন্দোলন কোথায় শুরু হয়েছিল ?", "১৯৫১", False),  # FIX 2 NEW: where->year blocked
    ("কোথায় বাংলাদেশের রাজধানী ?", "ঢাকা", True),
    ("বাংলাদেশ জামায়াতে ইসলামী কবে প্রতিষ্ঠিত হয় ?", "১৯৭৯ সালের মে মাসে", True),
]
print("\n  type-gate self-tests:")
all_ok = True
for q, g, want in _gate_tests:
    got = answer_type_ok(q, g)
    ok = got == want
    all_ok = all_ok and ok
    print(f"    {'OK ' if ok else 'FAIL'} gate({q[:45]!r}, {g[:20]!r}) -> {got} (want {want})")
assert all_ok, "type-gate self-tests failed — abort"

# ── FIX 1: RELATION-SPAN EXTRACTOR FOR CONTEXT VERIFICATION ─────
#
# The core insight from the error analysis:
# "Does the year appear in the context?" ≠ "Does the context say this IS the answer?"
#
# We map prompt question-words to RELATION KEYWORDS that should appear
# *near* the numeric answer in the relevant context sentence.

_RELATION_PATTERNS = {
    # Birth / death / formation
    "জন্ম": re.compile(r"জন্ম(?:গ্রহণ|িত|বার)?(?:\s|।)"),
    "মৃত্যু": re.compile(r"মৃত্যু(?:বরণ|বার)?(?:\s|।)|মারা যান|মৃত্যু হয়"),
    # Founding / establishment
    "প্রতিষ্ঠা": re.compile(r"প্রতিষ্ঠ(?:িত|া|র)?(?:\s|।)|স্থাপ(?:িত|ন|া)?(?:\s|।)"),
    # Construction start / end
    "নির্মাণ শুরু": re.compile(r"নির্মাণ\s*শুরু|নির্মাণ\s*আরম্ভ|নির্মাণকাজ\s*শুরু"),
    "নির্মাণ শেষ": re.compile(r"নির্মাণ\s*(?:সমাপ্ত|শেষ|সম্পন্ন|সমাপ্তি)"),
    # Recognition / award
    "স্বীকৃতি": re.compile(r"স্বীকৃতি|ইউনেস্কো|ঐতিহ্য(?:\s|।)"),
    "জাতীয় মর্যাদা": re.compile(r"জাতীয়\s*মর্যাদা|মর্যাদা\s*লাভ"),
    # Band / group formation
    "গঠন": re.compile(r"গঠ(?:িত|ন|িত\s*হয়)|দল\s*গঠন"),
    # Publication
    "প্রকাশ": re.compile(r"প্রকাশ(?:িত|ের)?(?:\s|।)|প্রথম\s*প্রকাশ"),
    # Independence / liberation
    "স্বাধীনতা": re.compile(r"স্বাধীনতা|মুক্তিযুদ্ধ"),
    # Discovery
    "আবিষ্কার": re.compile(r"আবিষ্কৃত|আবিষ্কার"),
}

# Map prompt keywords to which relation to check
_PROMPT_TO_RELATION = [
    (re.compile(r"কবে\s*জন্ম|জন্মগ্রহণ|জন্ম তারিখ"),            "জন্ম"),
    (re.compile(r"কবে\s*মৃত্যু|মৃত্যুবরণ|মৃত্যু\s*তারিখ"),      "মৃত্যু"),
    (re.compile(r"কত\s*সালে\s*প্রতিষ্ঠ|কবে\s*প্রতিষ্ঠ|কবে\s*স্থাপ"), "প্রতিষ্ঠা"),
    (re.compile(r"নির্মাণ\s*শুরু|নির্মাণকাজ\s*শুরু"),             "নির্মাণ শুরু"),
    (re.compile(r"নির্মাণ\s*শেষ|নির্মাণ\s*সমাপ্ত"),              "নির্মাণ শেষ"),
    (re.compile(r"ইউনেস্কো|বিশ্ব\s*ঐতিহ্য|স্বীকৃতি"),           "স্বীকৃতি"),
    (re.compile(r"জাতীয়\s*মর্যাদা"),                              "জাতীয় মর্যাদা"),
    (re.compile(r"দল.*গঠন|ব্যান্ড.*গঠন|গঠিত\s*হয়"),              "গঠন"),
    (re.compile(r"প্রকাশিত|প্রথম\s*প্রকাশ|কবে\s*লেখা"),          "প্রকাশ"),
    (re.compile(r"স্বাধীনতা\s*লাভ|মুক্তিযুদ্ধ\s*শুরু"),           "স্বাধীনতা"),
    (re.compile(r"আবিষ্কৃত|আবিষ্কার\s*হয়"),                      "আবিষ্কার"),
]

def get_prompt_relation(prompt: str):
    """Return the relation key for this prompt, or None if unknown."""
    for pattern, rel in _PROMPT_TO_RELATION:
        if pattern.search(prompt):
            return rel
    return None

def extract_relation_sentences(context: str, relation_key: str) -> list:
    """
    FIX 1: Extract only the sentences from context that contain the
    relation trigger for the prompted event type.
    Returns list of matching sentences (could be empty).
    """
    if not relation_key or relation_key not in _RELATION_PATTERNS:
        return []
    rel_re = _RELATION_PATTERNS[relation_key]
    # Split on sentence boundaries (। or newline)
    sentences = re.split(r"[।\n]+", context)
    return [s.strip() for s in sentences if rel_re.search(s) and len(s.strip()) > 5]

def relation_span_contains_response(context: str, prompt: str, response: str) -> tuple:
    """
    Returns (decision, confidence, tag):
      - If prompt has a known relation AND the relation-restricted sentences
        contain the response -> (1, high_conf, "L3_relation_match")
      - If prompt has a known relation AND relation sentences exist but
        do NOT contain the response -> (0, high_conf, "L3_relation_mismatch")
      - If prompt has a known relation but NO relation sentences found ->
        (None, 0, "L3_relation_absent") — pass to generic check
      - If no relation detected -> (None, 0, "") — pass to generic check
    """
    rel_key = get_prompt_relation(prompt)
    if not rel_key:
        return None, 0, ""

    rel_sentences = extract_relation_sentences(context, rel_key)
    if not rel_sentences:
        # Relation keyword not found in context — might be present under a
        # different phrasing; fall through to generic L3
        return None, 0, "L3_relation_absent"

    # Check if the response appears in ANY of the relation sentences
    nresp = bn_norm_numeric(response)
    combined = " ".join(rel_sentences)
    ncombined = bn_norm_numeric(combined)

    # Numeric check: any number in response must appear in relation span
    nums_resp = set(_NUM_RE.findall(nresp))
    nums_span = set(_NUM_RE.findall(ncombined))
    if nums_resp and nums_span:
        if nums_resp <= nums_span:
            return 1, 0.92, "L3_relation_match"
        else:
            return 0, 0.90, "L3_relation_mismatch"

    # Non-numeric: token containment
    if nresp and nresp in ncombined:
        return 1, 0.88, "L3_relation_match"
    core_r = core_tokens(response)
    core_s = core_tokens(combined)
    if core_r and core_s and len(core_r & core_s) / max(len(core_r), 1) >= 0.7:
        return 1, 0.80, "L3_relation_match"

    return 0, 0.85, "L3_relation_mismatch"

# ── S3. LAYER 1 — SAMPLE ANSWER-KEY MATCH ────────────────────────
log("S3 — Layer 1: sample answer-key")

sample_df["p_norm"] = sample_df["prompt_bn"].apply(bn_norm)
sample_df["r_norm"] = sample_df["response_bn"].apply(bn_norm)
test_df["p_norm"]   = test_df["prompt_bn"].apply(bn_norm)
test_df["r_norm"]   = test_df["response_bn"].apply(bn_norm)

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                      max_features=200_000, sublinear_tf=True)
S = sk_normalize(vec.fit_transform(sample_df["p_norm"]))
T = sk_normalize(vec.transform(test_df["p_norm"]))
sims = (T @ S.T).toarray()
top_i = sims.argmax(axis=1)
top_s = sims.max(axis=1)

N = len(test_df)
pred   = np.full(N, -1, dtype=int)
layer  = np.array([""] * N, dtype=object)
conf   = np.zeros(N, dtype=float)
hint   = np.array([""] * N, dtype=object)

n_gold_agree = n_gold_disagree = n_wrong_agree = n_pair = 0
for i in range(N):
    if top_s[i] < CFG["T1_SAMPLE_MATCH"]:
        continue
    srow = sample_df.iloc[top_i[i]]
    s_label = int(srow["label"])
    t_resp  = test_df["response_bn"].iat[i]
    s_resp  = srow["response_bn"]
    # L1a exact pair
    if test_df["p_norm"].iat[i] == srow["p_norm"] and test_df["r_norm"].iat[i] == srow["r_norm"]:
        pred[i], layer[i], conf[i] = s_label, "L1a_exact_pair", 0.99
        n_pair += 1
        continue
    # L1b
    if s_label == 1:
        agree = resp_agree(t_resp, s_resp)
        pred[i]  = 1 if agree else 0
        layer[i] = "L1b_gold_agree" if agree else "L1b_gold_disagree"
        conf[i]  = 0.96 if agree else 0.93
        n_gold_agree += int(agree); n_gold_disagree += int(not agree)
    else:
        if resp_agree(t_resp, s_resp):
            pred[i], layer[i], conf[i] = 0, "L1b_wrong_agree", 0.95
            n_wrong_agree += 1
        else:
            hint[i] = f"KNOWN-WRONG ANSWER for this question: {s_resp[:120]}"

print(f"  matched >= {CFG['T1_SAMPLE_MATCH']}: {(top_s>=CFG['T1_SAMPLE_MATCH']).sum()}")
print(f"  decided: exact-pair {n_pair} | gold-agree->1 {n_gold_agree} | "
      f"gold-disagree->0 {n_gold_disagree} | wrong-agree->0 {n_wrong_agree}")
print(f"  Layer-1 total decided: {(pred>=0).sum()} / {N}")

# ── S4. LAYER 2 — SQUAD_BN GOLD ANSWERS ──────────────────────────
log("S4 — Layer 2: squad_bn gold answers")

squad_q, squad_a = [], []
def _harvest(ds_iter, limit):
    got = 0
    for row in ds_iter:
        try:
            q = str(row.get("question", "")).strip()
            ans = row.get("answers", {})
            texts = ans.get("text", []) if isinstance(ans, dict) else []
            a = str(texts[0]).strip() if len(texts) else ""
            if q and a:
                squad_q.append(q); squad_a.append(a); got += 1
                if got >= limit: break
        except Exception:
            continue
    return got

SQUAD_OK = False

try:
    local_hits = []
    for root in CANDIDATE_ROOTS:
        if root.exists():
            local_hits += [p for p in root.rglob("*squad*")
                           if p.suffix in (".json", ".jsonl", ".parquet")]
    for p in local_hits[:6]:
        try:
            if p.suffix == ".parquet":
                d = pd.read_parquet(p)
                rows = d.to_dict("records")
            else:
                with open(p, encoding="utf-8") as f:
                    obj = json.load(f)
                rows = []
                if isinstance(obj, dict) and "data" in obj:
                    for art in obj["data"]:
                        for para in art.get("paragraphs", []):
                            for qa in para.get("qas", []):
                                ans = qa.get("answers", [])
                                rows.append({
                                    "question": qa.get("question", ""),
                                    "answers": {"text": [a.get("text","") for a in ans]},
                                })
                elif isinstance(obj, list):
                    rows = obj
            got = _harvest(iter(rows), CFG["SQUAD_MAX_ROWS"] - len(squad_q))
            print(f"  local {p.name}: +{got} QA pairs")
        except Exception as e:
            print(f"  local {p.name} failed: {str(e)[:80]}")
    if squad_q:
        SQUAD_OK = True
        print(f"  Route 0 (local files): {len(squad_q)} QA pairs total")
except Exception as e:
    print(f"  local squad search failed: {e}")

if not SQUAD_OK:
    try:
        import urllib.request, tarfile
        SQUAD_URL = ("https://huggingface.co/datasets/csebuetnlp/squad_bn/"
                     "resolve/main/data/squad_bn.tar.bz2")
        arc = WORK / "squad_bn.tar.bz2"
        ext = WORK / "squad_bn_extracted"
        if not arc.exists():
            log(f"  downloading squad_bn ...")
            urllib.request.urlretrieve(SQUAD_URL, arc)
        if not ext.exists():
            with tarfile.open(arc, "r:bz2") as tf:
                tf.extractall(ext)
        json_files = sorted(ext.rglob("*.json"))
        for p in json_files:
            with open(p, encoding="utf-8") as f:
                obj = json.load(f)
            rows = []
            for art in obj.get("data", []):
                for para in art.get("paragraphs", []):
                    for qa in para.get("qas", []):
                        ans = qa.get("answers", [])
                        if not ans: continue
                        rows.append({
                            "question": qa.get("question", ""),
                            "answers": {"text": [a.get("text", "") for a in ans]},
                        })
            got = _harvest(iter(rows), CFG["SQUAD_MAX_ROWS"] - len(squad_q))
            print(f"  {p.name}: +{got}")
        SQUAD_OK = len(squad_q) > 0
    except Exception as e:
        print(f"  squad_bn direct download failed: {e}")

if not SQUAD_OK:
    try:
        from datasets import load_dataset
        for split in ("train", "validation"):
            for kwargs in ({"streaming": True}, {"trust_remote_code": True}):
                try:
                    ds = load_dataset("csebuetnlp/squad_bn", split=split, **kwargs)
                    got = _harvest(iter(ds), CFG["SQUAD_MAX_ROWS"] - len(squad_q))
                    print(f"  squad_bn:{split} -> +{got}")
                    break
                except Exception as e:
                    print(f"  squad_bn:{split} failed: {str(e)[:80]}")
        SQUAD_OK = len(squad_q) > 0
    except Exception as e:
        print(f"  squad_bn load_dataset failed: {e}")

if SQUAD_OK:
    log(f"  indexing {len(squad_q)} squad_bn questions ...")
    sq_norm = [bn_norm(q) for q in squad_q]
    vec2 = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                           max_features=300_000, sublinear_tf=True)
    Q = sk_normalize(vec2.fit_transform(sq_norm))
    T2 = sk_normalize(vec2.transform(test_df["p_norm"]))

    BLOCK = 128
    l2_decided = 0
    for s in range(0, N, BLOCK):
        e = min(s + BLOCK, N)
        block_sims = (T2[s:e] @ Q.T).toarray()
        bi = block_sims.argmax(axis=1)
        bs = block_sims.max(axis=1)
        for k in range(e - s):
            i = s + k
            if pred[i] >= 0: continue
            # FIX 4: Use relaxed threshold for language route
            thr = CFG["T2_LANG_MATCH"] if test_df["route"].iat[i] == "language" else CFG["T2_SQUAD_MATCH"]
            if bs[k] < thr: continue
            gold = squad_a[bi[k]]
            q_text = test_df["prompt_bn"].iat[i]
            # FIX 2: type gate before accepting
            if not answer_type_ok(q_text, gold):
                hint[i] = (hint[i] + " | " if hint[i] else "") + f"NEAR-MATCH GOLD (type-mismatch): {gold[:100]}"
                continue
            agree = resp_agree(test_df["response_bn"].iat[i], gold)
            pred[i]  = 1 if agree else 0
            layer[i] = "L2_squad_agree" if agree else "L2_squad_disagree"
            conf[i]  = 0.90 if agree else 0.82
            if not hint[i]:
                hint[i] = f"SQUAD GOLD ANSWER: {gold[:120]}"
            l2_decided += 1
    print(f"  Layer-2 decided: {l2_decided}")

    # Calibration
    S2 = sk_normalize(vec2.transform(sample_df["p_norm"]))
    sample_sims = (S2 @ Q.T).toarray()
    si = sample_sims.argmax(axis=1); ss = sample_sims.max(axis=1)
    hit = ss >= CFG["T2_SQUAD_MATCH"]
    if hit.sum():
        correct = 0
        for j in np.where(hit)[0]:
            gold = squad_a[si[j]]
            if not answer_type_ok(sample_df["prompt_bn"].iloc[j], gold):
                continue
            p = 1 if resp_agree(sample_df["response_bn"].iloc[j], gold) else 0
            correct += int(p == int(sample_df["label"].iloc[j]))
        print(f"  L2 CALIBRATION: matched {hit.sum()}/299, acc {correct}/{hit.sum()} = {correct/hit.sum()*100:.1f}%")
else:
    print("  !! squad_bn unavailable — Layer 2 skipped")

# ── S5. LAYER 3 — TRACK-A WITH RELATION-SPAN FIX ─────────────────
log("S5 — Layer 3: Track-A with relation-span verification (FIX 1)")

_BN_DIGIT = re.compile(r"\d{2,}")

def a_signals(ctx, resp, q):
    nctx, nresp = bn_norm_numeric(ctx), bn_norm_numeric(resp)
    exact_sub = len(nresp) >= 2 and nresp in nctx
    w_r = set(nresp.split()); w_c = set(nctx.split())
    overlap = len(w_r & w_c) / max(len(w_r), 1)
    rd = set(_BN_DIGIT.findall(nresp)); cd = set(_BN_DIGIT.findall(nctx))
    digit_miss = bool(rd - cd) if rd else False
    year_q = bool(re.search(r"কত\s*সালে|কোন\s*সালে", q))
    type_mis = year_q and not bool(re.search(r"\d{4}", nresp))
    return exact_sub, overlap, digit_miss, type_mis

def layer3(ctx, resp, q):
    # FIX 2: type gate first
    # (already handled in the calling loop for year/who/where — belt & suspenders)
    # FIX 1: RELATION-SPAN CHECK FIRST for date questions
    rel_decision, rel_conf, rel_tag = relation_span_contains_response(ctx, resp, q)
    if rel_decision is not None and rel_tag not in ("L3_relation_absent",):
        return rel_decision, rel_conf, rel_tag

    # Generic fallback (non-date or relation not found in context)
    exact_sub, overlap, digit_miss, type_mis = a_signals(ctx, resp, q)
    if digit_miss: return 0, 0.85, "L3_digit_mismatch"
    if type_mis:   return 0, 0.70, "L3_type_mismatch"
    if exact_sub:  return 1, 0.85, "L3_exact_substring"
    if overlap >= CFG["A_OVERLAP_FAITH"]:
        return 1, 0.70, "L3_overlap_high"
    return 0, 0.55, "L3_overlap_low"

# CALIBRATE on sample Track A
from sklearn.metrics import f1_score, classification_report
A = sample_df[sample_df.has_ctx]
cal_pred = [layer3(r["ctx"], r["response_bn"], r["prompt_bn"])[0] for _, r in A.iterrows()]
acc = (np.array(cal_pred) == A["label"].values).mean()
print(f"  L3 CALIBRATION on sample Track-A ({len(A)} rows): acc={acc*100:.1f}%  "
      f"F1(hallucinated)={f1_score(A['label'], cal_pred, pos_label=0):.3f}")
print(classification_report(A["label"], cal_pred,
                            target_names=["hallucinated(0)", "faithful(1)"]))

# Print relation-specific calibration
date_rows = A[A["route"] == "date"]
if len(date_rows):
    dp = [layer3(r["ctx"], r["response_bn"], r["prompt_bn"])[0] for _, r in date_rows.iterrows()]
    da = (np.array(dp) == date_rows["label"].values).mean()
    print(f"  L3 CALIBRATION (date-route only, {len(date_rows)} rows): acc={da*100:.1f}%  "
          f"F1(hal)={f1_score(date_rows['label'], dp, pos_label=0, zero_division=0):.3f}")

l3_decided = 0
for i in range(N):
    if pred[i] >= 0 or not test_df["has_ctx"].iat[i]:
        continue
    p, c, tag = layer3(test_df["ctx"].iat[i], test_df["response_bn"].iat[i],
                       test_df["prompt_bn"].iat[i])
    pred[i], conf[i], layer[i] = p, c, tag
    l3_decided += 1
print(f"  Layer-3 decided: {l3_decided}")

# ── S6. ASSEMBLE OUTPUTS ─────────────────────────────────────────
log("S6 — assembling outputs")

undecided_mask = pred < 0
print(f"  decided by L1/L2/L3: {(~undecided_mask).sum()} | "
      f"undecided (LLM's job): {undecided_mask.sum()}")

# FIX 6/8: L2_squad_disagree rows are NOT simply set to 0.
# They are re-classified as undecided (with their squad hint) so the
# LLM judge can make a row-specific decision.  Only L3_overlap_low and
# genuine FALLBACK rows use the prior.
l2_dis_mask = np.array(layer == "L2_squad_disagree")
l2_dis_reset = 0
for i in np.where(l2_dis_mask)[0]:
    # Send to LLM: mark as undecided
    pred[i] = -1
    layer[i] = "L2_disagree_to_llm"
    l2_dis_reset += 1
print(f"  FIX 6: L2_squad_disagree rows re-routed to LLM: {l2_dis_reset}")

undecided_mask = pred < 0
# Fallback prior ONLY for rows that are still undecided AND NOT Track-A (context available)
# Track-A undecided rows get L3 or LLM; Track-B undecided go to LLM.
# We set a placeholder here so submission.csv is complete, but the LLM cell
# will overwrite these.
final = pred.copy()
for i in np.where(undecided_mask)[0]:
    final[i] = ROUTE_PRIOR[test_df["route"].iat[i]]
    if not layer[i]:
        layer[i] = f"FALLBACK_prior_{test_df['route'].iat[i]}"
    conf[i] = 0.5

layer_counts = Counter(layer)
print("\n  decision layer census:")
for k, v in sorted(layer_counts.items(), key=lambda kv: -kv[1]):
    print(f"    {k:30s} {v:5d}")
print(f"\n  prediction distribution: {pd.Series(final).value_counts().to_dict()}")

sub = pd.DataFrame({"id": test_df["id"], "label": final.astype(int)})
assert len(sub) == N and sub["label"].isin([0, 1]).all()
sub.to_csv(WORK / "submission.csv", index=False)

decisions = test_df[["id", "prompt_bn", "response_bn", "has_ctx", "route"]].copy()
decisions["pred"] = final
decisions["layer"] = layer
decisions["confidence"] = conf
decisions["llm_hint"] = hint
decisions["sample_match_sim"] = top_s
decisions.to_csv(WORK / "layer_decisions.csv", index=False)

undec = decisions[undecided_mask | l2_dis_mask].copy()
undec.to_csv(WORK / "undecided_for_llm.csv", index=False)

report = {
    "decided_L1": int(sum(v for k, v in layer_counts.items() if k.startswith("L1"))),
    "decided_L2": int(sum(v for k, v in layer_counts.items() if k.startswith("L2") and "disagree" not in k)),
    "decided_L3": int(sum(v for k, v in layer_counts.items() if k.startswith("L3") and k != "L3_overlap_low")),
    "fallback":   int(sum(v for k, v in layer_counts.items() if k.startswith("FALLBACK"))),
    "undecided_for_llm": int(undecided_mask.sum()),
    "l2_disagree_to_llm": l2_dis_reset,
    "config": CFG,
}
import json as _json
with open(WORK / "layer_report.json", "w") as f:
    _json.dump(report, f, indent=2)

print(f"\n  saved: submission.csv / layer_decisions.csv / undecided_for_llm.csv")
log("DONE with deterministic layers.")


CONFIG: {'T1_SAMPLE_MATCH': 0.9, 'T2_SQUAD_MATCH': 0.88, 'T2_LANG_MATCH': 0.82, 'A_OVERLAP_FAITH': 0.6, 'SQUAD_MAX_ROWS': 150000, 'SHORT_ANSWER_TOKS': 3}
[+    0s] S0 — loading data
  sample: /kaggle/input/competitions/bengali-hallucination/dataset samples.json
  test:   /kaggle/input/competitions/bengali-hallucination/test set.csv
  sample (299, 6) | test (2516, 6) | test Track-A 1361 / Track-B 1155
[+    0s] S1 — normalizers
  resp_agree self-tests:
    OK  agree('১০০ : ১০০.৬','১০০ : ১০০.৩') -> False (want False)
    OK  agree('০.০৯২৯','৬.৪৫') -> False (want False)
    OK  agree('চুরুলিয়া, বর্ধমান','টুঙ্গিপাড়া, ফরিদপুর') -> False (want False)
    OK  agree('স্বাধীন বাংলাদেশের প্রথম চলচ্চ','সুকুমারী') -> True (want True)
    OK  agree('৭ বছর','৬') -> False (want False)
    OK  agree("আঠারশ' বত্রিশ সালে",'১৮৩২') -> True (want True)
    OK  agree('মেহদী হাসান খান','মেহদী হাসান খান') -> True (want True)
    OK  agree('বৃহস্পতিবার','শুক্রবার') -> False (want False)
    OK  agree('১৯১৩',

/tmp/ipykernel_24/765431915.py:644: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(ext)


  test.json: +1252
  train.json: +68671
  validation.json: +1251
[+    5s]   indexing 71174 squad_bn questions ...
  Layer-2 decided: 854
  L2 CALIBRATION: matched 118/299, acc 108/118 = 91.5%
[+   27s] S5 — Layer 3: Track-A with relation-span verification (FIX 1)
  L3 CALIBRATION on sample Track-A (130 rows): acc=83.1%  F1(hallucinated)=0.725
                 precision    recall  f1-score   support

hallucinated(0)       0.88      0.62      0.72        47
    faithful(1)       0.81      0.95      0.88        83

       accuracy                           0.83       130
      macro avg       0.85      0.78      0.80       130
   weighted avg       0.84      0.83      0.82       130

  L3 CALIBRATION (date-route only, 34 rows): acc=85.3%  F1(hal)=0.783
  Layer-3 decided: 455
[+   27s] S6 — assembling outputs
  decided by L1/L2/L3: 1425 | undecided (LLM's job): 1091
  FIX 6: L2_squad_disagree rows re-routed to LLM: 330

  decision layer census:
    FALLBACK_prior_factual           668
   

## Cell 4 — LLM Engine, RAG, Question Bank & Judge Prompts

Sets up the LLM inference backend and all prompt templates used by the judge.

### LLM Backend
Tries model options in order, falling back gracefully:
1. `Qwen2.5-32B-Instruct-AWQ` via vLLM (TP=2)
2. `Qwen2.5-14B-Instruct-AWQ` via vLLM (TP=1)
3. `Qwen2.5-7B-Instruct-AWQ` via vLLM (TP=1)
4. `Qwen2.5-14B-Instruct` via HuggingFace 4-bit (fallback)

### RAG Corpus
BM25 retrieval over Bengali Wikipedia (up to 250k chunked passages, 160 words/chunk, 30-word overlap). Indexed with a BanglaBERT subword tokeniser for better recall on Bengali morphology.

### Prompt Templates

| Template | Purpose |
|----------|---------|
| `SYS_DATE_JUDGE` | Verifies that the candidate date matches the *specific event* asked, not just any date in the passage |
| `SYS_LANG_JUDGE` | Bengali linguist persona — handles antonyms, idioms, sandhi, samasa, prefix classes |
| `SYS_MATH` | Step-by-step solver with day-of-week modulo, sign handling, profit/loss, interest formulas |
| `SYS_FACTUAL_JUDGE` | Closed-book judge instructed that absence from the retrieval bank ≠ hallucination |
| `SYS_LANG_ANSWER` | Answer-first mode for language questions — generates the correct answer independently, then compares |

### Self-Consistency Voting
All LLM calls use `n=3` samples at temperature 0.7. Decisions require a majority vote; ties fall back to the question-bank hint or the route prior.


In [4]:
import json, os, re, sys, time, unicodedata, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

T0 = time.time()
def log(msg=""):
    print(f"[+{time.time()-T0:6.0f}s] {msg}", flush=True)

WORK = Path("/kaggle/working")

CFG2 = dict(
    TIME_BUDGET_H     = 6.0,
    MAX_MODEL_LEN     = 1536,
    JUDGE_MAX_TOKENS  = 200,    # increased for relation reasoning
    MATH_MAX_TOKENS   = 512,
    LANG_MAX_TOKENS   = 160,
    GPU_MEM_UTIL      = 0.88,
    SC_N              = 3,
    SC_TEMP           = 0.7,
    RAG_MAX_PASSAGES  = 250_000,
    RAG_TOP_K         = 3,
    QB_DECIDE_SIM     = 0.92,
    QB_HINT_SIM       = 0.80,
)
print("CONFIG:", CFG2)

# ── S0. LOAD DECISIONS + SAMPLE ──────────────────────────────────
log("S0 — loading layer_decisions.csv + sample")

DEC_CANDS = [WORK / "layer_decisions.csv"] + \
    list(Path("/kaggle/input").rglob("layer_decisions.csv"))
DEC_PATH = next((p for p in DEC_CANDS if p.exists()), None)
assert DEC_PATH, "layer_decisions.csv not found — run Cell 3 first"
dec = pd.read_csv(DEC_PATH)
dec["llm_hint"] = dec["llm_hint"].fillna("")
print(f"  decisions: {dec.shape}\n{dec['layer'].value_counts().to_string()}")

SAMPLE_CANDS = [p for pat in ("dataset samples (1).json", "dataset samples.json")
                for p in Path("/kaggle/input").rglob(pat)]
assert SAMPLE_CANDS, "sample json not found"
with open(SAMPLE_CANDS[0], encoding="utf-8") as f:
    sample_df = pd.DataFrame(json.load(f))

_NULL_TOKENS = {"[NULL]", "nan", "None", "", "null", "NULL"}
def norm_ctx(x):
    s = str(x).strip()
    return "" if s in _NULL_TOKENS else s

sample_df["ctx"] = sample_df["context"].apply(norm_ctx)
sample_df["has_ctx"] = sample_df["ctx"].ne("")
sample_df["prompt_bn"] = sample_df["prompt_bn"].astype(str)
sample_df["response_bn"] = sample_df["response_bn"].astype(str)

# ── S1. NORMALIZERS (copy from Cell 3) ───────────────────────────
BN2ASCII = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
_BN_NUM_WORDS = {
    "শূন্য":0,"এক":1,"দুই":2,"দু":2,"তিন":3,"চার":4,"পাঁচ":5,"ছয়":6,
    "সাত":7,"আট":8,"নয়":9,"দশ":10,"বিশ":20,"ত্রিশ":30,"চল্লিশ":40,
    "পঞ্চাশ":50,"ষাট":60,"সত্তর":70,"আশি":80,"নব্বই":90,
    "শত":100,"হাজার":1000,"লক্ষ":100000,"লাখ":100000,"কোটি":10000000,
}
_SHO_RE = re.compile(r"([\u0980-\u09FF]+?)(?:শো|শ)(?=\s|$|[^\u0980-\u09FF])")

def bn_words_to_digits(text):
    t = re.sub(r"['''`]", "", str(text))
    def _sho(m):
        base = _BN_NUM_WORDS.get(m.group(1))
        return str(base * 100) if base and base < 100 else m.group(0)
    t = _SHO_RE.sub(_sho, t)
    toks = t.split(); out, acc, has_acc = [], 0, False
    def _flush():
        nonlocal acc, has_acc
        if has_acc: out.append(str(acc)); acc, has_acc = 0, False
    for w in toks:
        wc = w.strip("।,.;:!\"()")
        v = _BN_NUM_WORDS.get(wc) if wc in _BN_NUM_WORDS else (int(wc) if wc.isdigit() else None)
        if v is None: _flush(); out.append(w); continue
        if not has_acc: acc, has_acc = v, True
        elif v in (100,1000,100000,10000000): acc *= v
        elif v < 100 and acc >= 100 and acc % 100 == 0: acc += v
        else: _flush(); acc, has_acc = v, True
    _flush()
    return " ".join(out)

def bn_norm(text):
    t = unicodedata.normalize("NFC", str(text)).translate(BN2ASCII)
    t = re.sub(r"\*\*|__|`", " ", t)
    t = re.sub(r'[!@#$%^&*()_+={}|;<>?,/~`]', ' ', t)
    t = re.sub(r'[\[\]\-\\]', ' ', t)
    t = re.sub('[\"\u201c\u201d\u2018\u2019]', ' ', t)
    t = re.sub('[\u0964\u2013\u2014]', ' ', t)
    return re.sub(r"\s+", " ", t).strip().lower()

def bn_norm_numeric(text):
    return bn_norm(bn_words_to_digits(str(text)))

_NUM_RE = re.compile(
    r"(?<![\w\d])[+-]?\d+(?:\.\d+)?"
    r"(?:\s*/\s*[+-]?\d+(?:\.\d+)?)?"
)

def numbers_of(text):
    raw = unicodedata.normalize("NFC", bn_words_to_digits(str(text)))
    raw = raw.translate(BN2ASCII).lower()
    return tuple(re.sub(r"\s+", "", m.group(0)) for m in _NUM_RE.finditer(raw))

_POSITIVE_RE = re.compile(
    r"\b(?:positive|plus)\b|ধনাত্মক|ইতিবাচক|পজিটিভ",
    re.I | re.U,
)
_NEGATIVE_RE = re.compile(
    r"\b(?:negative|minus)\b|ঋণাত্মক|নেতিবাচক|নেগেটিভ",
    re.I | re.U,
)

def polarity_of(text):
    raw = unicodedata.normalize("NFC", str(text)).lower()
    has_pos = bool(_POSITIVE_RE.search(raw))
    has_neg = bool(_NEGATIVE_RE.search(raw))
    if has_pos and not has_neg:
        return 1
    if has_neg and not has_pos:
        return -1
    return None

_STOP = set("এর কে কি কী কোন কত হয় হলো ছিল করে থেকে একটি টি জন সালে সাল "
            "নাম বলা মোট প্রায় হচ্ছে এবং বা ও the a an of in is was".split())
def core_tokens(text):
    return {w for w in bn_norm_numeric(text).split() if w not in _STOP and len(w) > 1}

def resp_agree(a, b):
    na, nb = bn_norm_numeric(a), bn_norm_numeric(b)
    if not na or not nb:
        return False

    # Check structured numeric content before normalized equality because bn_norm
    # intentionally removes signs and fraction punctuation.
    num_a, num_b = numbers_of(a), numbers_of(b)
    if num_a and num_b:
        if num_a != num_b:
            return False

        pol_a, pol_b = polarity_of(a), polarity_of(b)
        if (pol_a == -1 and pol_b != -1) or (pol_b == -1 and pol_a != -1):
            return False
        if pol_a is not None and pol_b is not None and pol_a != pol_b:
            return False
        return True

    if na == nb:
        return True

    toks_a = na.split()
    toks_b = nb.split()
    short = min(len(toks_a), len(toks_b)) <= 3
    if short:
        return (na in nb or nb in na) and (len(na) >= 2 and len(nb) >= 2)
    if len(na) >= 3 and len(nb) >= 3 and (na in nb or nb in na):
        return True

    ca, cb = core_tokens(a), core_tokens(b)
    if not ca or not cb:
        return False
    return len(ca & cb) / len(ca | cb) >= 0.5

DATE_RE = re.compile(r"কত\s*সালে|কোন\s*সালে|কবে\s")
LANG_RE = re.compile(
    r"অর্থ\s*কী|ভাবার্থ|সন্ধি|সমাস|বানান.*শুদ্ধ|শুদ্ধ.*বানান|"
    r"প্রতিশব্দ|বিপরীত\s*শব্দ|ব্যাকরণ|শব্দের\s*অর্থ|"
    r"কারক|বিভক্তি|প্রত্যয়|উপসর্গ|পদ\s*কী|বাগধারা|প্রবাদ|"
    r"বাগধারার অর্থ|শব্দের ভাবার্থ|সমাস নির্ণয়|সন্ধি বিচ্ছেদ|উপসর্গের শ্রেণি|"
    r"শব্দটির অর্থ|কোন শ্রেণির")
MATH_RE = re.compile(
    r"কত[টি]?\s*(হবে|হয়|হওয়ার)|যোগফল|বিয়োগফল|গুণফল|ভাগফল|"
    r"শতকরা|সুদে|সুদের হার|লাভ.*ক্ষতি|ক্ষতি.*লাভ|ক্রয়.*বিক্রয়|"
    r"\d+\s*[\+\-\×\÷\*\/]|সমীকরণ|অনুপাত|লসাগু|গসাগু|"
    r"কোন বার|কোন দিন|বার হবে|বয়স কত|চক্রবৃদ্ধি সুদ|বর্গমূল|"
    r"একটি মিশ্রণে|শতাংশ হ্রাস|শতাংশ বৃদ্ধি")

def route_of(p):
    if MATH_RE.search(p): return "math"
    if LANG_RE.search(p): return "language"
    if DATE_RE.search(p): return "date"
    return "factual"

sample_df["route"] = sample_df["prompt_bn"].apply(route_of)

PRIOR = {}
for (r, hc), g in sample_df.groupby(["route", "has_ctx"]):
    PRIOR[(r, bool(hc))] = dict(p1=g["label"].mean(), n=len(g),
                                maj=int(g["label"].mean() >= 0.5))
print("  (route, has_ctx) -> P(label=1):")
for k, v in sorted(PRIOR.items()):
    print(f"    {str(k):22s} p1={v['p1']:.2f}  n={v['n']:3d}  maj={v['maj']}")

def prior_pred(route, has_ctx):
    v = PRIOR.get((route, bool(has_ctx)))
    if v is None: v = dict(maj=int(sample_df["label"].mean() >= 0.5))
    return v["maj"]

# ── S2. WORKLOAD ──────────────────────────────────────────────────
log("S2 — building LLM workload")

REJUDGE_LAYERS = {"L3_overlap_low", "L2_disagree_to_llm"}
work_mask = dec["layer"].str.startswith("FALLBACK") | dec["layer"].isin(REJUDGE_LAYERS)
work = dec[work_mask].copy()
work["is_math"] = work["route"].eq("math")
print(f"  rows to judge: {len(work)} (math {work['is_math'].sum()}, other {len(work)-work['is_math'].sum()})")

TEST_CANDS = [p for pat in ("test set (3).csv", "test set.csv")
              for p in Path("/kaggle/input").rglob(pat)]
test_ctx = {}
if TEST_CANDS:
    tdf = pd.read_csv(TEST_CANDS[0])
    if "id" not in tdf.columns:
        tdf = tdf.reset_index().rename(columns={"index": "id"})
    for _, r in tdf.iterrows():
        test_ctx[int(r["id"])] = norm_ctx(r.get("context", ""))

# ── S3. LLM ENGINE ────────────────────────────────────────────────
log("S3 — loading LLM")

ENGINE = None
MODEL_USED = "none"

VLLM_CHAIN = [
    ("Qwen/Qwen2.5-32B-Instruct-AWQ", 2),
    ("Qwen/Qwen2.5-14B-Instruct-AWQ", 1),
    ("Qwen/Qwen2.5-7B-Instruct-AWQ",  1),
]

def try_vllm():
    global ENGINE, MODEL_USED
    try:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"],
                       check=False, timeout=1200)
        os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
        from vllm import LLM
        from transformers import AutoTokenizer
        import torch
        n_gpu = torch.cuda.device_count()
        for mid, tp in VLLM_CHAIN:
            if tp > n_gpu: print(f"  skip {mid} (needs TP={tp})"); continue
            try:
                log(f"  vLLM loading {mid} ...")
                big = "32B" in mid
                llm = LLM(model=mid, quantization="awq", dtype="half",
                          tensor_parallel_size=tp,
                          gpu_memory_utilization=CFG2["GPU_MEM_UTIL"],
                          max_model_len=CFG2["MAX_MODEL_LEN"],
                          max_num_seqs=48 if big else 128,
                          enforce_eager=big)
                tok = AutoTokenizer.from_pretrained(mid)
                ENGINE = ("vllm", llm, tok)
                MODEL_USED = mid
                log(f"  vLLM ready: {mid}")
                return True
            except Exception as e:
                print(f"  vLLM {mid} failed: {str(e)[:150]}")
                gc.collect()
                try: import torch; torch.cuda.empty_cache()
                except: pass
        return False
    except Exception as e:
        print(f"  vLLM unavailable: {str(e)[:150]}")
        return False

def try_hf():
    global ENGINE, MODEL_USED
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        if not torch.cuda.is_available(): return False
        mid = "Qwen/Qwen2.5-14B-Instruct"
        log(f"  HF 4-bit loading {mid} ...")
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                 bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16)
        tok = AutoTokenizer.from_pretrained(mid)
        tok.padding_side = "left"
        if tok.pad_token is None: tok.pad_token = tok.eos_token
        mod = AutoModelForCausalLM.from_pretrained(mid, quantization_config=bnb, device_map="auto")
        mod.eval()
        ENGINE = ("hf", mod, tok)
        MODEL_USED = mid + " (4bit)"
        log(f"  HF ready: {MODEL_USED}")
        return True
    except Exception as e:
        print(f"  HF fallback failed: {str(e)[:150]}")
        return False

HAVE_LLM = try_vllm() or try_hf()
if not HAVE_LLM:
    print("  !! NO LLM — falling back to priors only")

def chat_wrap(system, user):
    kind, _, tok = ENGINE
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

SC_SEP = "\n###SC###\n"

def generate(prompts, max_tokens, n=1):
    kind, eng, tok = ENGINE
    if kind == "vllm":
        from vllm import SamplingParams
        sp = SamplingParams(n=n, temperature=CFG2["SC_TEMP"] if n > 1 else 0.0,
                            top_p=0.9, max_tokens=max_tokens)
        outs = eng.generate(prompts, sp)
        return [SC_SEP.join(c.text for c in o.outputs) for o in outs]
    else:
        import torch
        res = []
        for s in range(0, len(prompts), 8):
            batch = prompts[s:s+8]
            enc = tok(batch, return_tensors="pt", padding=True,
                      truncation=True, max_length=CFG2["MAX_MODEL_LEN"]).to("cuda")
            with torch.no_grad():
                out = eng.generate(**enc, max_new_tokens=max_tokens,
                                   do_sample=False, pad_token_id=tok.eos_token_id)
            in_len = enc["input_ids"].shape[1]
            res += [tok.decode(o[in_len:], skip_special_tokens=True) for o in out]
        return res

# ── S3.5 RAG ─────────────────────────────────────────────────────
log("S3.5 — RAG corpus + BM25")

import pickle, subprocess
RAG_OK = False
bm25 = None
rag_passages = []
CORPUS_CACHE = WORK / "rag_corpus.pkl"
BM25_CACHE   = WORK / "rag_bm25_subword.pkl"

def chunk_text(text, size=160, overlap=30):
    words = str(text).split()
    return [" ".join(words[i:i+size])
            for i in range(0, len(words), size - overlap)
            if len(words[i:i+size]) >= 20]

try:
    from transformers import AutoTokenizer as _AT
    _bn_tok = _AT.from_pretrained("csebuetnlp/banglabert")
    def bn_tokenize(text):
        return _bn_tok.tokenize(bn_norm(text))[:256]
except Exception:
    def bn_tokenize(text):
        return bn_norm(text).split()

try:
    if CORPUS_CACHE.exists():
        rag_passages = pickle.load(open(CORPUS_CACHE, "rb"))
        print(f"  corpus cache: {len(rag_passages)} passages")
    else:
        from datasets import load_dataset
        wiki = load_dataset("wikimedia/wikipedia", "20231101.bn",
                            split="train", streaming=True)
        for art in wiki:
            rag_passages.extend(chunk_text(art.get("text", "")))
            if len(rag_passages) >= CFG2["RAG_MAX_PASSAGES"]: break
        pickle.dump(rag_passages, open(CORPUS_CACHE, "wb"))
        print(f"  corpus built: {len(rag_passages)} passages")
    if BM25_CACHE.exists():
        bm25 = pickle.load(open(BM25_CACHE, "rb"))
        print("  BM25 cache loaded")
    else:
        from rank_bm25 import BM25Okapi
        log(f"  tokenizing {len(rag_passages)} passages ...")
        _toks = [bn_tokenize(p) for p in rag_passages]
        log("  building BM25 ...")
        bm25 = BM25Okapi(_toks)
        pickle.dump(bm25, open(BM25_CACHE, "wb"))
        del _toks
    RAG_OK = bm25 is not None and len(rag_passages) > 0
except Exception as e:
    print(f"  RAG unavailable ({str(e)[:120]})")

def retrieve_evidence(query, k=None, extra=""):
    if not RAG_OK: return []
    k = k or CFG2["RAG_TOP_K"]
    scores = np.asarray(bm25.get_scores(bn_tokenize(str(query) + " " + str(extra))))
    idx = np.argsort(scores)[::-1][:k]
    return [rag_passages[i][:280] for i in idx if scores[i] > 0]

# ── S3.6 QUESTION BANK ───────────────────────────────────────────
log("S3.6 — question-bank index")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize

try:
    qb_q, qb_a
    print(f"  preserving {len(qb_q)} pre-loaded QA pairs")
except NameError:
    qb_q, qb_a = [], []

def _opt_text(row, ans):
    opts = None
    for c in ("options", "choices", "option_list"):
        if c in row and isinstance(row[c], (list, tuple)) and row[c]:
            opts = list(row[c]); break
    if opts is None:
        letters = [row.get(f"option_{l}") or row.get(f"option{l.upper()}") for l in "abcd"]
        opts = [x for x in letters if x] or None
    a = str(ans).strip()
    if opts:
        if a.isdigit() and int(a) < len(opts) + 1:
            i = int(a)
            return str(opts[i] if i < len(opts) else opts[i - 1])
        LM = {"a":0,"b":1,"c":2,"d":3,"e":4,"ক":0,"খ":1,"গ":2,"ঘ":3,"ঙ":4}
        if a.lower() in LM and LM[a.lower()] < len(opts):
            return str(opts[LM[a.lower()]])
    return a

def _add_qb_rows(rows):
    added = 0
    for row in rows:
        q = next((str(row[c]).strip() for c in ("question","question_bn","prompt","query") if row.get(c)), None)
        if not q: continue
        ans = next((row[c] for c in ("answer","answer_key","correct_answer","answer_bn","correct","label","ans") if row.get(c) is not None), None)
        if ans is None: continue
        a_text = _opt_text(row, ans)
        if a_text and len(str(a_text)) >= 1:
            qb_q.append(q); qb_a.append(str(a_text)); added += 1
    return added

try:
    from datasets import load_dataset
    for split in ("test", "validation", "dev", "train"):
        try:
            ds = load_dataset("hishab/bangla-mmlu", split=split)
            n = _add_qb_rows(list(ds))
            print(f"  bangla-mmlu:{split} -> +{n} (total {len(qb_q)})")
        except Exception as e:
            print(f"  bangla-mmlu:{split}: {str(e)[:80]}")
except Exception as e:
    print(f"  bangla-mmlu unavailable: {e}")

for p in list(Path("/kaggle/input").rglob("*.csv"))[:40]:
    if any(x in p.name for x in ("layer_decisions", "test set", "submission")): continue
    try:
        d = pd.read_csv(p, nrows=200000)
        cols = {c.lower() for c in d.columns}
        if cols & {"question","question_bn"} and cols & {"answer","correct_answer","answer_key","ans","answer_bn"}:
            d.columns = [c.lower() for c in d.columns]
            n = _add_qb_rows(d.to_dict("records"))
            print(f"  attached {p.name}: +{n}")
    except Exception: pass

QB_OK = len(qb_q) > 500
qb_vec = qb_M = None
if QB_OK:
    log(f"  indexing {len(qb_q)} QB questions ...")
    qb_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                             max_features=300_000, sublinear_tf=True)
    qb_M = sk_normalize(qb_vec.fit_transform([bn_norm(q) for q in qb_q]))
    print("  QB index ready")
else:
    print("  !! QB disabled (not enough QA pairs)")

def _top1_sparse(M, bank, block=256):
    ti = np.empty(M.shape[0], dtype=np.int64); ts = np.empty(M.shape[0], dtype=np.float32)
    for s in range(0, M.shape[0], block):
        sims = (M[s:s+block] @ bank.T).toarray()
        ti[s:s+block] = sims.argmax(1); ts[s:s+block] = sims.max(1)
    return ti, ts

def qb_lookup(questions):
    if not QB_OK: return [(0.0, "")] * len(questions)
    Tq = sk_normalize(qb_vec.transform([bn_norm(q) for q in questions]))
    out = []
    for s in range(0, Tq.shape[0], 256):
        sims = (Tq[s:s+256] @ qb_M.T).toarray()
        bi = sims.argmax(axis=1); bs = sims.max(axis=1)
        out += [(float(bs[k]), qb_a[bi[k]]) for k in range(len(bi))]
    return out

# ── S4. IMPROVED PROMPTS ─────────────────────────────────────────
#
# PROMPT FIX A: Relation-aware date judge
SYS_DATE_JUDGE = (
    "You are a strict Bengali fact-checker specializing in DATE and EVENT questions.\n"
    "You will receive a Bengali question asking about a SPECIFIC DATE/YEAR for a SPECIFIC EVENT "
    "(e.g. founding year, birth year, construction start year, recognition year).\n"
    "A passage may mention MULTIPLE dates. Your job is to verify that the candidate answer\n"
    "is the date for the EXACT EVENT asked about — not just any date from the passage.\n\n"
    "Rules:\n"
    "1. Read the question carefully: what SPECIFIC EVENT'S date is being asked?\n"
    "2. Find the sentence(s) in the context/evidence that describe THAT specific event.\n"
    "3. If the candidate answer matches the date in THOSE sentences -> CORRECT.\n"
    "4. If the candidate answer is a date from a DIFFERENT event in the passage -> WRONG.\n"
    "5. Think in 2-4 sentences, then output:\n"
    "VERDICT: CORRECT   or   VERDICT: WRONG\n"
    "CONFIDENCE: <1-10>"
)

# PROMPT FIX B: Bengali language/grammar judge
SYS_LANG_JUDGE = (
    "You are an expert Bengali linguist. You will receive a Bengali linguistics question\n"
    "and a candidate answer. Your task is to verify if the candidate answer is linguistically correct.\n\n"
    "Rules:\n"
    "1. Identify the linguistic task: antonym (বিপরীত শব্দ), idiom meaning (বাগধারার অর্থ),\n"
    "   prefix classification (উপসর্গের শ্রেণি), compound word analysis (সমাস), etc.\n"
    "2. For ANTONYMS: accept any standard antonym, not just one specific form.\n"
    "3. For IDIOM MEANINGS: accept semantically equivalent paraphrases.\n"
    "4. For PREFIX CLASSES: তৎসম, তদ্ভব, বিদেশি etc. — verify the correct class.\n"
    "5. For সমাস: verify the exact compound type (কর্মধারয়, বহুব্রীহি, দ্বন্দ্ব, দ্বিগু, তৎপুরুষ).\n"
    "6. Think in 2-3 sentences, then output:\n"
    "VERDICT: CORRECT   or   VERDICT: WRONG\n"
    "CONFIDENCE: <1-10>"
)

# PROMPT FIX C: Improved math solver
SYS_MATH = (
    "You are a careful Bengali math solver. Solve the problem step by step.\n"
    "Convert Bengali numerals to digits. Handle:\n"
    "- Percentages (শতকরা/শতাংশ): as fractions\n"
    "- Days of week: count modulo 7 (রবিবার=0, সোমবার=1, মঙ্গলবার=2, বুধবার=3, বৃহস্পতিবার=4, শুক্রবার=5, শনিবার=6)\n"
    "- Profit/loss (লাভ/ক্ষতি): (SP-CP)/CP × 100\n"
    "- Simple interest: P×R×T/100\n"
    "- Compound interest: P×(1+R/100)^T\n"
    "- Negative fractions: include the sign\n"
    "- Algebraic equations: solve for the variable exactly\n"
    "Show steps, then end with exactly:\n"
    "FINAL: <numeric answer only, with sign/units if needed>"
)

# PROMPT FIX D: Closed-book factual judge (FIX for Track-B false negatives)
SYS_FACTUAL_JUDGE = (
    "You are a strict Bengali fact-checker. You will get a Bengali question\n"
    "and a candidate answer. Decide if the answer is FACTUALLY CORRECT.\n\n"
    "IMPORTANT: The question has NO supplied context. Use your knowledge and any\n"
    "evidence passages provided. DO NOT assume a correct answer is wrong just because\n"
    "it wasn't found in a database — well-known facts (Padma Bridge, historical figures,\n"
    "publications, dates) may be correct even if a retrieval system missed them.\n\n"
    "Bangladesh-specific facts often differ from global defaults — be careful.\n"
    "Think briefly in 2-3 sentences, then output:\n"
    "VERDICT: CORRECT   or   VERDICT: WRONG\n"
    "CONFIDENCE: <1-10>"
)

SYS_ANSWER = (
    "You are a Bengali knowledge expert. Answer the question concisely and factually.\n"
    "Bangladesh-specific facts often differ from global defaults.\n"
    "If evidence passages are given, prefer them when relevant.\n"
    "End with exactly:\n"
    "FINAL: <your answer in Bengali, as short as possible>"
)

# PROMPT FIX B2: Language answer-first mode
SYS_LANG_ANSWER = (
    "You are an expert Bengali linguist. Answer this Bengali linguistics question.\n"
    "Give the linguistically correct answer only.\n"
    "End with exactly:\n"
    "FINAL: <correct answer in Bengali>"
)

def judge_prompt(q, resp, hint="", ctx="", evidence=None, route="factual"):
    # Select system prompt based on route and track
    if ctx:
        if route == "date":
            sys = SYS_DATE_JUDGE  # FIX A
        else:
            sys = SYS_FACTUAL_JUDGE
    elif route == "language":
        sys = SYS_LANG_JUDGE      # FIX B
    elif route == "date":
        sys = SYS_DATE_JUDGE
    else:
        sys = SYS_FACTUAL_JUDGE   # FIX D

    u = ""
    if ctx:
        u += f"Context (ground truth passage):\n{ctx[:900]}\n\n"
    elif evidence:
        ev = "\n".join(f"[{i+1}] {p}" for i, p in enumerate(evidence))
        u += f"Evidence (may or may not be relevant):\n{ev}\n\n"
    u += f"Question: {q}\nCandidate answer: {resp}\n"
    if hint:
        u += f"\nNote: {hint}\n"
    if ctx and route == "date":
        u += "\nJudge ONLY whether the candidate date matches the SPECIFIC EVENT asked about in the question."
    elif ctx:
        u += "\nJudge ONLY against the context above."
    elif evidence:
        u += "\nUse the evidence when relevant; otherwise use your own knowledge."
    return chat_wrap(sys, u)

def math_prompt(q):
    return chat_wrap(SYS_MATH, f"Problem: {q}")

def answer_prompt(q, evidence=None, route="factual"):
    sys = SYS_LANG_ANSWER if route == "language" else SYS_ANSWER
    u = ""
    if evidence:
        ev = "\n".join(f"[{i+1}] {p}" for i, p in enumerate(evidence))
        u += f"Evidence (may or may not be relevant):\n{ev}\n\n"
    u += f"Question: {q}"
    return chat_wrap(sys, u)

_VERDICT_RE = re.compile(r"VERDICT\s*:\s*(CORRECT|WRONG)", re.I)
_CONF_RE    = re.compile(r"CONFIDENCE\s*:\s*(\d+)", re.I)
_FINAL_RE   = re.compile(r"FINAL\s*:\s*(.+)", re.I)

def parse_judge(text):
    m = _VERDICT_RE.search(text or "")
    v = m.group(1).upper() if m else None
    c = _CONF_RE.search(text or "")
    conf = min(10, max(1, int(c.group(1)))) if c else 5
    return v, conf

def vote_judge(text):
    votes = [v for part in (text or "").split(SC_SEP) for v, _ in [parse_judge(part)] if v]
    if not votes: return None
    c = Counter(votes)
    if len(c) > 1 and c.most_common(1)[0][1] == c.most_common(2)[1][1]: return None
    return c.most_common(1)[0][0]

def vote_answer(text, resp):
    votes = []
    for part in (text or "").split(SC_SEP):
        m = _FINAL_RE.search(part or "")
        if m: votes.append(1 if resp_agree(m.group(1).strip(), resp) else 0)
    if not votes: return None
    c = Counter(votes)
    if len(c) > 1 and c.most_common(1)[0][1] == c.most_common(2)[1][1]: return None
    return c.most_common(1)[0][0]

def vote_math(text, resp):
    votes = []
    for part in (text or "").split(SC_SEP):
        m = _FINAL_RE.search(part or "")
        if not m: continue
        solved = m.group(1).strip()
        ns, nr = numbers_of(solved), numbers_of(resp)
        if ns and nr:
            votes.append(1 if ns == nr else 0)
        else:
            votes.append(1 if resp_agree(solved, resp) else 0)
    if not votes: return None
    c = Counter(votes)
    if len(c) > 1 and c.most_common(1)[0][1] == c.most_common(2)[1][1]: return None
    return c.most_common(1)[0][0]

# ── S5. CACHE ─────────────────────────────────────────────────────
CACHE_PATH = WORK / "llm_cache.csv"
cache = {}
if CACHE_PATH.exists():
    cdf = pd.read_csv(CACHE_PATH)
    cache = dict(zip(cdf["key"], cdf["text"].fillna("")))
    print(f"  cache loaded: {len(cache)} generations")

def cached_generate(keys, prompts, max_tokens, n=1):
    todo = [(k, p) for k, p in zip(keys, prompts) if k not in cache]
    log(f"  generating {len(todo)}/{len(keys)} (rest cached, n={n}) ...")
    if todo and HAVE_LLM:
        CHUNK = 64
        for s in range(0, len(todo), CHUNK):
            part = todo[s:s+CHUNK]
            outs = generate([p for _, p in part], max_tokens, n=n)
            for (k, _), o in zip(part, outs):
                cache[k] = o
            pd.DataFrame({"key": list(cache.keys()),
                          "text": list(cache.values())}).to_csv(CACHE_PATH, index=False)
            log(f"    {min(s+CHUNK, len(todo))}/{len(todo)} done, cache saved")
    return [cache.get(k, "") for k in keys]

# ── S6. THROUGHPUT PROBE ──────────────────────────────────────────
if HAVE_LLM:
    log("S6 — throughput probe")
    probe = [judge_prompt("বাংলাদেশের রাজধানী কোথায়?", "ঢাকা")] * 8
    t0 = time.time()
    _ = generate(probe, 64)
    dt = time.time() - t0
    per_row = dt / 8
    n_total = len(work) + (~sample_df.has_ctx).sum() + 20
    proj_h = per_row * n_total / 3600 * 1.6
    log(f"  probe: {per_row:.2f}s/row -> projected {proj_h:.1f}h (budget {CFG2['TIME_BUDGET_H']}h)")
    if proj_h > CFG2["TIME_BUDGET_H"] and ENGINE[0] == "vllm" and "32B" in MODEL_USED:
        print("  !! projection exceeds budget — reload with 14B")
        del ENGINE; ENGINE = None; gc.collect()
        VLLM_CHAIN[:] = VLLM_CHAIN[1:]
        HAVE_LLM = try_vllm() or try_hf()

# ── S7. CALIBRATE JUDGE ON SAMPLE ────────────────────────────────
JUDGE_MODE_BY_ROUTE = {}
if HAVE_LLM:
    log("S7 — calibrating judge on sample (per-route mode selection)")
    calB = sample_df[~sample_df.has_ctx].copy()

    if QB_OK:
        hits = qb_lookup(calB["prompt_bn"].tolist())
        qb_dec, qb_ok = 0, 0
        for (s, gold), (_, r) in zip(hits, calB.iterrows()):
            if s >= CFG2["QB_DECIDE_SIM"]:
                p = 1 if resp_agree(str(r["response_bn"]), gold) else 0
                qb_dec += 1
                qb_ok += int(p == int(r["label"]))
        print(f"  QB CALIBRATION: decided {qb_dec}/{len(calB)}, acc {qb_ok}/{max(qb_dec,1)} = "
              f"{qb_ok/max(qb_dec,1)*100:.1f}%")

    kv, ka, meta_other = [], [], []
    km, meta_m = [], []
    kl, kla, meta_l = [], [], []

    for idx, r in calB.iterrows():
        rte = r["route"]
        ev = retrieve_evidence(r["prompt_bn"]) if rte in ("factual", "date") else []
        if rte == "math":
            km.append(f"cal_math_v3_{idx}")
            meta_m.append((idx, r))
        elif rte == "language":
            kl.append(f"cal_lang_j_v3_{idx}")
            kla.append(f"cal_lang_a_v3_{idx}")
            meta_l.append((idx, r))
        else:
            kv.append(f"cal_judge_v3_{idx}")
            ka.append(f"cal_ans_v3_{idx}")
            meta_other.append((idx, r))

    # Generate for math
    outs_m = cached_generate(km,
        [math_prompt(str(calB.loc[idx, "prompt_bn"])) for idx, _ in meta_m],
        CFG2["MATH_MAX_TOKENS"], n=CFG2["SC_N"])

    # Generate for language — both verdict and answer modes
    outs_l_j = cached_generate(kl,
        [judge_prompt(str(r["prompt_bn"]), str(r["response_bn"]), route="language") for _, r in meta_l],
        CFG2["LANG_MAX_TOKENS"], n=CFG2["SC_N"])
    outs_l_a = cached_generate(kla,
        [answer_prompt(str(r["prompt_bn"]), route="language") for _, r in meta_l],
        CFG2["LANG_MAX_TOKENS"], n=CFG2["SC_N"])

    # Generate for factual/date
    outs_v = cached_generate(kv,
        [judge_prompt(str(r["prompt_bn"]), str(r["response_bn"]),
                      evidence=retrieve_evidence(str(r["prompt_bn"])),
                      route=str(r["route"])) for _, r in meta_other],
        CFG2["JUDGE_MAX_TOKENS"], n=CFG2["SC_N"])
    outs_a = cached_generate(ka,
        [answer_prompt(str(r["prompt_bn"]),
                       evidence=retrieve_evidence(str(r["prompt_bn"])),
                       route=str(r["route"])) for _, r in meta_other],
        CFG2["JUDGE_MAX_TOKENS"], n=CFG2["SC_N"])

    from collections import defaultdict
    stats = defaultdict(lambda: dict(n=0, prior=0, verdict=0, answer=0))

    for (idx, r), ov, oa in zip(meta_other, outs_v, outs_a):
        rte, y = r["route"], int(r["label"])
        st = stats[rte]; st["n"] += 1
        st["prior"] += int(prior_pred(rte, False) == y)
        v = vote_judge(ov)
        pv_ = (1 if v == "CORRECT" else 0) if v is not None else prior_pred(rte, False)
        st["verdict"] += int(pv_ == y)
        a_vote = vote_answer(oa, str(r["response_bn"]))
        pa_ = a_vote if a_vote is not None else prior_pred(rte, False)
        st["answer"] += int(pa_ == y)

    st = stats["language"]
    for (idx, r), oj, oa in zip(meta_l, outs_l_j, outs_l_a):
        y = int(r["label"]); st["n"] += 1
        st["prior"] += int(prior_pred("language", False) == y)
        v = vote_judge(oj)
        pv_ = (1 if v == "CORRECT" else 0) if v is not None else prior_pred("language", False)
        st["verdict"] += int(pv_ == y)
        a_vote = vote_answer(oa, str(r["response_bn"]))
        pa_ = a_vote if a_vote is not None else prior_pred("language", False)
        st["answer"] += int(pa_ == y)

    st = stats["math"]
    for (idx, r), om in zip(meta_m, outs_m):
        y = int(r["label"]); st["n"] += 1
        st["prior"] += int(prior_pred("math", False) == y)
        p = vote_math(om, str(r["response_bn"]))
        pm = p if p is not None else prior_pred("math", False)
        st["verdict"] += int(pm == y); st["answer"] += int(pm == y)

    print("\n  per-route mode selection (accuracy on sample null rows):")
    for rte, s in stats.items():
        n = max(s["n"], 1)
        accs = {m: s[m]/n for m in ("prior", "verdict", "answer")}
        best = max(accs, key=accs.get)
        JUDGE_MODE_BY_ROUTE[rte] = best
        print(f"    {rte:10s} n={s['n']:3d}  prior={accs['prior']*100:5.1f}%  "
              f"verdict={accs['verdict']*100:5.1f}%  answer={accs['answer']*100:5.1f}%"
              f"  -> {best.upper()}")
else:
    JUDGE_MODE_BY_ROUTE = {r: "prior" for r in ("math","language","date","factual")}


CONFIG: {'TIME_BUDGET_H': 6.0, 'MAX_MODEL_LEN': 1536, 'JUDGE_MAX_TOKENS': 200, 'MATH_MAX_TOKENS': 512, 'LANG_MAX_TOKENS': 160, 'GPU_MEM_UTIL': 0.88, 'SC_N': 3, 'SC_TEMP': 0.7, 'RAG_MAX_PASSAGES': 250000, 'RAG_TOP_K': 3, 'QB_DECIDE_SIM': 0.92, 'QB_HINT_SIM': 0.8}
[+     0s] S0 — loading layer_decisions.csv + sample
  decisions: (2516, 10)
layer
FALLBACK_prior_factual     668
L2_squad_agree             524
L2_disagree_to_llm         330
L3_overlap_low             225
FALLBACK_prior_math        176
FALLBACK_prior_language    173
L3_exact_substring         151
L1b_gold_disagree           98
FALLBACK_prior_date         74
L3_overlap_high             57
L3_digit_mismatch           18
L1a_exact_pair              11
L1b_wrong_agree              4
L1b_gold_agree               3
L3_relation_mismatch         3
L3_type_mismatch             1
  (route, has_ctx) -> P(label=1):
    ('date', False)        p1=0.40  n=  5  maj=0
    ('date', True)         p1=0.62  n= 34  maj=1
    ('factual', False)    

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 07-17 20:13:43 [model.py:619] Resolved architecture: Qwen2ForCausalLM
INFO 07-17 20:13:43 [model.py:1776] Using max model len 1536
INFO 07-17 20:13:44 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Parse safetensors files:   0%|          | 0/5 [00:00<?, ?it/s]

...

Processed prompts: 100%|██████████| 192/192 [03:53<00:00,  1.22s/it, est. speed input: 811.63 toks/s, output: 64.08 toks/s]

[+  2312s]     64/127 done, cache saved


Rendering prompts:   0%|          | 0/63 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 189/189 [04:05<00:00,  1.30s/it, est. speed input: 782.55 toks/s, output: 64.10 toks/s]

[+  2558s]     127/127 done, cache saved

  per-route mode selection (accuracy on sample null rows):
    factual    n=122  prior= 50.8%  verdict= 64.8%  answer= 59.0%  -> VERDICT
    date       n=  5  prior= 60.0%  verdict= 60.0%  answer= 40.0%  -> PRIOR
    language   n= 31  prior= 54.8%  verdict= 71.0%  answer= 58.1%  -> VERDICT
    math       n= 11  prior= 63.6%  verdict= 72.7%  answer= 72.7%  -> VERDICT


## Cell 5 — Dense Embedder (BGE-M3)

Loads `BAAI/bge-m3` as a dense encoder for semantic similarity retrieval. Used alongside TF-IDF in Cells 7 and 8 to form a hybrid retrieval system.

- Primary: `sentence-transformers` wrapper
- Fallback: plain `transformers` CLS-pooling encoder if `sentence-transformers` is unavailable
- Embeddings are cached to disk to avoid re-encoding across runs


In [5]:
import hashlib, gc
import numpy as np, torch
from pathlib import Path
WORK = Path("/kaggle/working")

EMB_MODEL_ID = "BAAI/bge-m3"
EMB_OK = False
_emb = None

class PlainEncoder:
    def __init__(self, model_id, device="cuda"):
        from transformers import AutoTokenizer, AutoModel
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.mod = AutoModel.from_pretrained(model_id, torch_dtype=torch.float16).to(device).eval()
        self.device = device
        self.max_seq_length = 128
    @torch.no_grad()
    def encode(self, texts, batch_size=128, normalize_embeddings=True,
               convert_to_numpy=True, show_progress_bar=False):
        out = []
        for s in range(0, len(texts), batch_size):
            b = [str(t) for t in texts[s:s+batch_size]]
            enc = self.tok(b, padding=True, truncation=True,
                           max_length=self.max_seq_length, return_tensors="pt").to(self.device)
            h = self.mod(**enc).last_hidden_state[:, 0]
            if normalize_embeddings:
                h = torch.nn.functional.normalize(h, dim=-1)
            out.append(h.float().cpu().numpy())
        return np.concatenate(out, axis=0)

try:
    from sentence_transformers import SentenceTransformer
    _emb = SentenceTransformer(EMB_MODEL_ID, device="cuda" if torch.cuda.is_available() else "cpu")
    _emb.max_seq_length = 128
    EMB_OK = True
    print("embedder: sentence-transformers OK ->", EMB_MODEL_ID)
except Exception as e:
    print(f"sentence-transformers unusable ({type(e).__name__}: {str(e)[:100]})")
    try:
        _emb = PlainEncoder(EMB_MODEL_ID)
        EMB_OK = True
        print("embedder: FALLBACK plain-transformers encoder ->", EMB_MODEL_ID)
    except Exception as e2:
        print(f"fallback encoder ALSO failed ({str(e2)[:100]})")
        print("!! dense retrieval disabled — pipeline continues with TF-IDF-only")

def _corpus_key(texts, tag):
    h = hashlib.md5((str(len(texts)) + str(texts[0])[:80] + str(texts[-1])[:80]).encode()).hexdigest()[:10]
    return WORK / f"emb_{tag}_{h}.npy"

def encode_cached(texts, tag, batch=128):
    texts = [str(t) for t in texts]
    f = _corpus_key(texts, tag)
    if f.exists():
        E = np.load(f); print(f"  emb cache hit: {tag} {E.shape}"); return E
    E = _emb.encode(texts, batch_size=batch, normalize_embeddings=True,
                    convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
    np.save(f, E); print(f"  encoded {tag}: {E.shape} -> {f.name}")
    return E

def dense_top1(E_query, E_bank, block=512):
    ti = np.empty(len(E_query), dtype=np.int64)
    ts = np.empty(len(E_query), dtype=np.float32)
    for s in range(0, len(E_query), block):
        sims = E_query[s:s+block] @ E_bank.T
        ti[s:s+block] = sims.argmax(1); ts[s:s+block] = sims.max(1)
    return ti, ts

sentence-transformers unusable (RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your e)


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

fallback encoder ALSO failed (CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 4.)
!! dense retrieval disabled — pipeline continues with TF-IDF-only


## Cell 6 — Answer-Type Gate (Re-import)

Re-exports `answer_type_ok` for use by the hybrid retrieval cells that follow. This gate rejects gold answers whose semantic type does not match the question type before any agreement check is performed.

| Question type | Gate condition |
|---------------|---------------|
| Year (কত সালে) | Gold must contain a 4-digit year |
| Count (কতটি) | Gold must contain any digit |
| Who (কে) | Gold must not be a bare 4-digit number |
| Where (কোথায়) | Gold must not start with a 4-digit year |
| Other | Pass through |


In [6]:
import re

_YEAR_Q   = re.compile(r"কত\s*সালে|কোন\s*সালে|(?:^|\s)কবে(?:\s|$|[?।,])|কোন\s*বছর")
_COUNT_Q  = re.compile(r"কত[টি](?:\s|$|[?।,])|কয়টি|কতগুলো|কত\s*জন|সংখ্যা\s*কত")
_WHO_Q    = re.compile(r"(?:^|\s)কে(?:\s|$|[?।,])|কার\s*নাম|কাকে|কে\s*ছিলেন")
_WHERE_Q  = re.compile(r"(?:^|\s)কোথায়(?:\s|$|[?।,])|কোন স্থানে|কোন জায়গায়")
_4DIGIT   = re.compile(r"\b\d{4}\b")
_ANYNUM   = re.compile(r"\d")

def answer_type_ok(question: str, gold: str) -> bool:
    q = str(question)
    g = bn_norm_numeric(str(gold))
    if _YEAR_Q.search(q): return bool(_4DIGIT.search(g))
    if _COUNT_Q.search(q): return bool(_ANYNUM.search(g))
    if _WHO_Q.search(q):
        g_stripped = g.replace(" ", "")
        return not (g_stripped.isdigit() and len(g_stripped) >= 4)
    if _WHERE_Q.search(q): return not bool(_4DIGIT.match(g.strip()))
    return True

print("type-gate imported ok")

type-gate imported ok


## Cell 7 — Layer 2 v2: Hybrid Squad-BN Matching

Upgrades the Layer 2 squad-based retrieval to a **TF-IDF + dense (BGE-M3) union** retriever.

- TF-IDF threshold: 0.88
- Dense threshold: 0.92 (only activated for rows that TF-IDF misses)
- The answer-type gate is applied before accepting any matched gold answer
- A calibration sweep over the labelled sample set is printed to confirm accuracy before test predictions are written

Results are merged back into `layer_decisions.csv`.


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize
import numpy as np

L2_TFIDF_T = 0.88
L2_DENSE_T = 0.92

l2_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                         max_features=300_000, sublinear_tf=True)
l2_Q = sk_normalize(l2_vec.fit_transform([bn_norm(q) for q in squad_q]))
_sample_pn = sample_df["p_norm"] if "p_norm" in sample_df else sample_df["prompt_bn"].apply(bn_norm)
_test_pn   = test_df["p_norm"]   if "p_norm" in test_df   else test_df["prompt_bn"].apply(bn_norm)
l2_S = sk_normalize(l2_vec.transform(_sample_pn))
l2_T = sk_normalize(l2_vec.transform(_test_pn))

def _top1_sparse(M, bank, block=256):
    ti = np.empty(M.shape[0], dtype=np.int64); ts = np.empty(M.shape[0], dtype=np.float32)
    for s in range(0, M.shape[0], block):
        sims = (M[s:s+block] @ bank.T).toarray()
        ti[s:s+block] = sims.argmax(1); ts[s:s+block] = sims.max(1)
    return ti, ts

tf_i_s, tf_s_s = _top1_sparse(l2_S, l2_Q)
tf_i_t, tf_s_t = _top1_sparse(l2_T, l2_Q)

if EMB_OK:
    E_squad  = encode_cached(squad_q, "squad")
    E_sample = encode_cached(sample_df["prompt_bn"].tolist(), "sample_p")
    E_test   = encode_cached(test_df["prompt_bn"].tolist(), "test_p")
    dn_i_s, dn_s_s = dense_top1(E_sample, E_squad)
    dn_i_t, dn_s_t = dense_top1(E_test, E_squad)
else:
    dn_i_s = np.zeros(len(sample_df), dtype=np.int64); dn_s_s = np.zeros(len(sample_df), dtype=np.float32)
    dn_i_t = np.zeros(len(test_df), dtype=np.int64);   dn_s_t = np.zeros(len(test_df), dtype=np.float32)
    print("EMB_OK=False -> dense skipped (TF-IDF only)")

# Calibration sweep
y_cal = sample_df["label"].astype(int).values
resp_cal = sample_df["response_bn"].astype(str).tolist()
q_cal = sample_df["prompt_bn"].astype(str).tolist()

def _score(mask, idx):
    d = o = g = 0
    for j in np.where(mask)[0]:
        gold = squad_a[idx[j]]
        if not answer_type_ok(q_cal[j], gold): g += 1; continue
        p = 1 if resp_agree(resp_cal[j], gold) else 0
        d += 1; o += int(p == y_cal[j])
    return d, o, g

print(f"{'matcher':30s} {'decided':>7s} {'acc':>7s} {'gated':>6s}")
d, o, g = _score(tf_s_s >= L2_TFIDF_T, tf_i_s)
print(f"{'tfidf >= %.2f (+gate)' % L2_TFIDF_T:30s} {d:7d} {o/max(d,1)*100:6.1f}% {g:6d}")
if EMB_OK:
    for t in (0.85, 0.88, 0.90, 0.92, 0.95):
        d, o, g = _score((dn_s_s >= t) & (tf_s_s < L2_TFIDF_T), dn_i_s)
        print(f"{'dense-added >= %.2f (+gate)' % t:30s} {d:7d} {o/max(d,1)*100:6.1f}% {g:6d}")
    u_mask = (tf_s_s >= L2_TFIDF_T) | (dn_s_s >= L2_DENSE_T)
    u_idx  = np.where(tf_s_s >= L2_TFIDF_T, tf_i_s, dn_i_s)
    d, o, g = _score(u_mask, u_idx)
    print(f"{'UNION tf%.2f|dn%.2f (+gate)' % (L2_TFIDF_T, L2_DENSE_T):30s} {d:7d} {o/max(d,1)*100:6.1f}% {g:6d}")

# Apply to test
id2row = {int(r["id"]): i for i, r in dec.iterrows()}
n_new = n_regate = n_hint = 0
for k in range(len(test_df)):
    rid = int(test_df["id"].iat[k]); i = id2row[rid]
    cur = str(dec.at[i, "layer"])
    if cur.startswith("L1"): continue
    tf_hit = tf_s_t[k] >= L2_TFIDF_T
    dn_hit = bool(EMB_OK) and (dn_s_t[k] >= L2_DENSE_T)
    if not (tf_hit or dn_hit): continue
    gold = squad_a[tf_i_t[k] if tf_hit else dn_i_t[k]]
    q    = str(test_df["prompt_bn"].iat[k])
    resp = str(test_df["response_bn"].iat[k])
    if not answer_type_ok(q, gold):
        if cur.startswith("L2"):
            dec.at[i, "layer"] = "FALLBACK_prior_" + str(dec.at[i, "route"])
            dec.at[i, "pred"] = -1
            n_regate += 1
        h = f"NEAR-MATCH GOLD (type-mismatch, verify): {str(gold)[:100]}"
        old_h = str(dec.at[i, "llm_hint"] or "")
        if h not in old_h:
            dec.at[i, "llm_hint"] = (old_h + " | " if old_h else "") + h
        n_hint += 1
        continue
    p = 1 if resp_agree(resp, gold) else 0
    was_new = not cur.startswith("L2")
    dec.at[i, "pred"] = p
    dec.at[i, "layer"] = "L2v2_agree" if p == 1 else "L2v2_disagree"
    dec.at[i, "confidence"] = 0.93 if p == 1 else 0.90
    n_new += int(was_new)

print(f"\nL2 v2 applied: newly-claimed {n_new} | old-L2 re-gated {n_regate} | mismatch hints {n_hint}")
print(dec["layer"].value_counts().to_string())
dec.to_csv(WORK / "layer_decisions.csv", index=False)

EMB_OK=False -> dense skipped (TF-IDF only)
matcher                        decided     acc  gated
tfidf >= 0.88 (+gate)              115   93.0%      3

L2 v2 applied: newly-claimed 0 | old-L2 re-gated 0 | mismatch hints 35
layer
FALLBACK_prior_factual     668
L2v2_agree                 524
L2v2_disagree              330
L3_overlap_low             225
FALLBACK_prior_math        176
FALLBACK_prior_language    173
L3_exact_substring         151
L1b_gold_disagree           98
FALLBACK_prior_date         74
L3_overlap_high             57
L3_digit_mismatch           18
L1a_exact_pair              11
L1b_wrong_agree              4
L1b_gold_agree               3
L3_relation_mismatch         3
L3_type_mismatch             1


## Cell 8 — QB v2: Hybrid Exam-Bank Lookup

Upgrades the question-bank lookup to the same TF-IDF + dense hybrid used in Cell 7.

- Threshold: 0.92 for both TF-IDF and dense matchers
- Rows below the decide threshold but above 0.80 are kept as *hints* for the LLM judge
- Calibration is printed separately for TF-IDF and dense contributions


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize
import numpy as np

QB_TFIDF_T = 0.92
QB_DENSE_T = 0.92

qb_vec2 = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                          max_features=300_000, sublinear_tf=True)
qb_M2 = sk_normalize(qb_vec2.fit_transform([bn_norm(q) for q in qb_q]))
E_qb = encode_cached(qb_q, "qbank") if EMB_OK else None

def qb_lookup_hybrid(questions):
    qs = [str(q) for q in questions]
    Tq = sk_normalize(qb_vec2.transform([bn_norm(q) for q in qs]))
    tf_i, tf_s = _top1_sparse(Tq, qb_M2)
    if EMB_OK:
        Eq = _emb.encode(qs, batch_size=128, normalize_embeddings=True,
                         convert_to_numpy=True).astype(np.float32)
        dn_i, dn_s = dense_top1(Eq, E_qb)
    else:
        dn_i = np.zeros(len(qs), dtype=np.int64); dn_s = np.zeros(len(qs), dtype=np.float32)
    out = []
    for k in range(len(qs)):
        if tf_s[k] >= QB_TFIDF_T:
            out.append((float(tf_s[k]), qb_a[tf_i[k]], "tf"))
        elif EMB_OK and dn_s[k] >= QB_DENSE_T:
            out.append((float(dn_s[k]), qb_a[dn_i[k]], "dn"))
        elif max(tf_s[k], dn_s[k]) >= 0.80:
            best = (tf_i[k], tf_s[k]) if tf_s[k] >= dn_s[k] else (dn_i[k], dn_s[k])
            out.append((float(best[1]), qb_a[best[0]], "hint"))
        else:
            out.append((0.0, "", ""))
    return out

# Calibration
calB = sample_df[~sample_df["has_ctx"]].copy()
hits = qb_lookup_hybrid(calB["prompt_bn"].tolist())
st = {"tf": [0, 0], "dn": [0, 0]}; gated = 0
for (s, gold, src), (_, r) in zip(hits, calB.iterrows()):
    if src not in ("tf", "dn"): continue
    if not answer_type_ok(str(r["prompt_bn"]), gold): gated += 1; continue
    p = 1 if resp_agree(str(r["response_bn"]), gold) else 0
    st[src][0] += 1; st[src][1] += int(p == int(r["label"]))
tot_d = st["tf"][0] + st["dn"][0]; tot_o = st["tf"][1] + st["dn"][1]
print(f"QB v2 CALIBRATION on {len(calB)} null-ctx sample rows:")
print(f"  tfidf-decided {st['tf'][0]:3d}  acc {st['tf'][1]}/{max(st['tf'][0],1)} = {st['tf'][1]/max(st['tf'][0],1)*100:.1f}%")
print(f"  dense-added   {st['dn'][0]:3d}  acc {st['dn'][1]}/{max(st['dn'][0],1)} = {st['dn'][1]/max(st['dn'][0],1)*100:.1f}%")
print(f"  UNION         {tot_d:3d}  acc {tot_o}/{max(tot_d,1)} = {tot_o/max(tot_d,1)*100:.1f}% | gated: {gated}")

QB v2 CALIBRATION on 169 null-ctx sample rows:
  tfidf-decided  69  acc 63/69 = 91.3%
  dense-added     0  acc 0/1 = 0.0%
  UNION          69  acc 63/69 = 91.3% | gated: 0


## Cell 9 — LLM Judge Workload

Sends all remaining undecided rows to the LLM judge. Rows handled here are those labelled `FALLBACK`, `L3_overlap_low`, `L2_disagree_to_llm`, or `L2v2_disagree`.

**Per-route dispatch:**
- `math` → `SYS_MATH` step-by-step solver with self-consistency voting on numeric output
- `language` → both `SYS_LANG_JUDGE` (verdict) and `SYS_LANG_ANSWER` (answer-first) in parallel; answer-first mode is preferred for language when available
- `date` / `factual` with context → `SYS_DATE_JUDGE` / `SYS_FACTUAL_JUDGE` with the context passage
- `date` / `factual` without context → RAG evidence retrieved from Bengali Wikipedia + `SYS_FACTUAL_JUDGE`

All generations are cached to `llm_cache.csv` so the cell is safe to re-run.


In [9]:
from collections import Counter

work_mask2 = (
    dec["layer"].str.startswith("FALLBACK") |
    dec["layer"].isin(["L3_overlap_low", "L2_disagree_to_llm", "L2v2_disagree"])
)
work2 = dec[work_mask2].copy()
work2["is_math"] = work2["route"].eq("math")
print(f"workload: {len(work2)} rows | routes: {work2['route'].value_counts().to_dict()}")

final_pred2, final_src2 = {}, {}
qb_hint_map2 = {}

# a) QB v2 first claim
hits = qb_lookup_hybrid(work2["prompt_bn"].astype(str).tolist())
n_qb = 0
for (s, gold, src), (_, r) in zip(hits, work2.iterrows()):
    rid = int(r["id"]); q = str(r["prompt_bn"])
    if src in ("tf", "dn") and answer_type_ok(q, gold):
        final_pred2[rid] = 1 if resp_agree(str(r["response_bn"]), gold) else 0
        final_src2[rid] = f"QB2_{src}"
        n_qb += 1
    elif gold:
        qb_hint_map2[rid] = f"LIKELY GOLD ANSWER (exam bank, verify): {str(gold)[:120]}"
print(f"QB v2 decided: {n_qb} | hints: {len(qb_hint_map2)}")

work_left = work2[~work2["id"].astype(int).isin(final_pred2)].copy()
print(f"remaining for LLM: {len(work_left)}")

if HAVE_LLM and len(work_left):
    kv, pv, ka, pa, meta_fd = [], [], [], [], []
    kl, pl, kla, pla, meta_l = [], [], [], [], []
    km, pm, meta_m = [], [], []

    for _, r in work_left.iterrows():
        rid = int(r["id"]); rte = str(r["route"])
        hint = str(r.get("llm_hint", "") or "")
        if rid in qb_hint_map2:
            hint = (hint + " | " if hint else "") + qb_hint_map2[rid]
        ctx = test_ctx.get(rid, "") if str(r["layer"]).startswith("L3") else ""

        if rte == "math":
            km.append(f"test_math_v3_{rid}"); pm.append(math_prompt(str(r["prompt_bn"])))
            meta_m.append((rid, r))
        elif rte == "language":
            # FIX B: language gets both verdict and answer-first mode
            kl.append(f"test_lang_j_v3_{rid}")
            pl.append(judge_prompt(str(r["prompt_bn"]), str(r["response_bn"]),
                                   hint=hint, route="language"))
            kla.append(f"test_lang_a_v3_{rid}")
            pla.append(answer_prompt(str(r["prompt_bn"]), route="language"))
            meta_l.append((rid, r))
        else:  # factual + date pooled — FIX A and FIX D
            ev = [ctx[:800]] if ctx else retrieve_evidence(str(r["prompt_bn"]))
            kv.append(f"test_judge_v3_{rid}")
            pv.append(judge_prompt(str(r["prompt_bn"]), str(r["response_bn"]),
                                   hint=hint, ctx=ctx,
                                   evidence=None if ctx else ev,
                                   route=rte))
            ka.append(f"test_ans_v3_{rid}")
            pa.append(answer_prompt(str(r["prompt_bn"]),
                                    evidence=ev if not ctx else None,
                                    route=rte))
            meta_fd.append((rid, r))

    outs_m  = cached_generate(km,  pm,  CFG2["MATH_MAX_TOKENS"],  n=CFG2["SC_N"])
    outs_lj = cached_generate(kl,  pl,  CFG2["LANG_MAX_TOKENS"],  n=CFG2["SC_N"])
    outs_la = cached_generate(kla, pla, CFG2["LANG_MAX_TOKENS"],  n=CFG2["SC_N"])
    outs_v  = cached_generate(kv,  pv,  CFG2["JUDGE_MAX_TOKENS"], n=CFG2["SC_N"])
    outs_a  = cached_generate(ka,  pa,  CFG2["JUDGE_MAX_TOKENS"], n=CFG2["SC_N"])

    n_agree = n_tie_qb = n_tie_v = 0

    # factual + date ensemble
    for (rid, r), ov, oa in zip(meta_fd, outs_v, outs_a):
        rte, hc = str(r["route"]), bool(r["has_ctx"])
        v = vote_judge(ov)
        pv_ = (1 if v == "CORRECT" else 0) if v is not None else None
        pa_ = vote_answer(oa, str(r["response_bn"]))
        if pv_ is not None and pa_ is not None and pv_ == pa_:
            p, src = pv_, "ENS_agree"; n_agree += 1
        elif rid in qb_hint_map2:
            gold = qb_hint_map2[rid].split(":", 1)[-1].strip()
            p, src = (1 if resp_agree(str(r["response_bn"]), gold) else 0), "ENS_qb_tie"
            n_tie_qb += 1
        elif pv_ is not None:
            p, src = pv_, "ENS_verdict"; n_tie_v += 1
        elif pa_ is not None:
            p, src = pa_, "ENS_answer"
        else:
            p, src = prior_pred(rte, hc), "PRIOR"
        final_pred2[rid], final_src2[rid] = p, src

    # language — FIX B: verdict AND answer-first, ensemble
    for (rid, r), oj, oa in zip(meta_l, outs_lj, outs_la):
        hc = bool(r["has_ctx"])
        v = vote_judge(oj)
        pv_ = (1 if v == "CORRECT" else 0) if v is not None else None
        pa_ = vote_answer(oa, str(r["response_bn"]))
        if pv_ is not None and pa_ is not None and pv_ == pa_:
            p, src = pv_, "LANG_ENS_agree"
        elif rid in qb_hint_map2:
            gold = qb_hint_map2[rid].split(":", 1)[-1].strip()
            p, src = (1 if resp_agree(str(r["response_bn"]), gold) else 0), "LANG_QB_tie"
        elif pa_ is not None:
            # FIX B: prefer answer-first for language (better handles semantic equivalence)
            p, src = pa_, "LANG_answer"
        elif pv_ is not None:
            p, src = pv_, "LANG_verdict"
        else:
            p, src = prior_pred("language", hc), "PRIOR"
        final_pred2[rid], final_src2[rid] = p, src

    # math
    for (rid, r), o in zip(meta_m, outs_m):
        p = vote_math(o, str(r["response_bn"]))
        final_pred2[rid] = p if p is not None else prior_pred("math", bool(r["has_ctx"]))
        final_src2[rid] = "LLM_math" if p is not None else "PRIOR"

    print(f"factual/date ensemble: agree {n_agree} | QB tiebreak {n_tie_qb} | verdict tiebreak {n_tie_v}")
else:
    for _, r in work_left.iterrows():
        rid = int(r["id"])
        final_pred2[rid] = prior_pred(str(r["route"]), bool(r["has_ctx"]))
        final_src2[rid] = "PRIOR"
    print(f"no LLM: {len(work_left)} rows set by (route,track) priors")

print("src census:", Counter(final_src2.values()))

workload: 1646 rows | routes: {'factual': 936, 'language': 395, 'math': 178, 'date': 137}
QB v2 decided: 487 | hints: 100
remaining for LLM: 1159
[+  3520s]   generating 135/135 (rest cached, n=3) ...


Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 192/192 [03:59<00:00,  1.25s/it, est. speed input: 301.43 toks/s, output: 216.17 toks/s]

[+  3759s]     64/135 done, cache saved


Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]

...

Processed prompts: 100%|██████████| 192/192 [04:26<00:00,  1.39s/it, est. speed input: 750.23 toks/s, output: 64.34 toks/s]

[+ 10491s]     640/654 done, cache saved


Rendering prompts:   0%|          | 0/14 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 42/42 [00:58<00:00,  1.38s/it, est. speed input: 587.73 toks/s, output: 70.99 toks/s]


[+ 10549s]     654/654 done, cache saved
factual/date ensemble: agree 526 | QB tiebreak 27 | verdict tiebreak 101
src census: Counter({'ENS_agree': 526, 'QB2_tf': 487, 'LANG_ENS_agree': 210, 'LANG_answer': 153, 'LLM_math': 133, 'ENS_verdict': 101, 'ENS_qb_tie': 27, 'LANG_QB_tie': 6, 'PRIOR': 2, 'LANG_verdict': 1})


## Cell 10 — Merge, QB Cross-Exam & Final Submission

Merges all prediction sources and writes the final `submission.csv`.

1. Layer decisions from Cell 3 are used as the base
2. LLM judge outputs from Cell 9 override any prior placeholder predictions
3. **QB cross-examination:** rows labelled `L2v2_disagree` or `L1b_gold_disagree` are checked against the exam bank — if the exam bank confirms the response is correct, the label is flipped to 1
4. Any row still missing a prediction receives a safety prior based on its `(route, has_context)` bucket
5. Final layer census and prediction distribution are printed for inspection


In [10]:
import json as _json, time as _time
import pandas as pd

dec["final_pred"]  = dec["pred"]
dec["final_layer"] = dec["layer"]
id2row = {int(r["id"]): i for i, r in dec.iterrows()}
n_override = 0
for rid, p in final_pred2.items():
    i = id2row[rid]
    old = int(dec.at[i, "pred"])
    dec.at[i, "final_pred"]  = int(p)
    dec.at[i, "final_layer"] = final_src2.get(rid, "PRIOR2")
    n_override += int(old != int(p))
print(f"workload overrides that changed a prediction: {n_override}")

# QB cross-exam of disagree rows
targets = dec[dec["final_layer"].isin(["L2v2_disagree", "L1b_gold_disagree"])]
print(f"cross-examining {len(targets)} disagree-rows against exam bank")
flips = confirms = conflicts_on_agree = 0
if len(targets):
    hits = qb_lookup_hybrid(targets["prompt_bn"].astype(str).tolist())
    for (s, gold, src), (i, r) in zip(hits, targets.iterrows()):
        if src not in ("tf", "dn") or not answer_type_ok(str(r["prompt_bn"]), gold):
            continue
        if resp_agree(str(r["response_bn"]), gold):
            if int(dec.at[i, "final_pred"]) == 0:
                dec.at[i, "final_pred"] = 1
                dec.at[i, "final_layer"] = "QBX_" + str(r["final_layer"])
                flips += 1
        else:
            confirms += 1
ag = dec[dec["final_layer"] == "L2v2_agree"]
if len(ag):
    hits2 = qb_lookup_hybrid(ag["prompt_bn"].astype(str).tolist())
    for (s, gold, src), (_, r) in zip(hits2, ag.iterrows()):
        if src in ("tf", "dn") and answer_type_ok(str(r["prompt_bn"]), gold) \
           and not resp_agree(str(r["response_bn"]), gold):
            conflicts_on_agree += 1
print(f"QBX flips 0->1: {flips} | QB-confirmed 0s: {confirms} | "
      f"QB-vs-L2v2 conflicts on agree-rows: {conflicts_on_agree}")

# Safety sweep
left = dec["final_pred"].astype(int) < 0
for i in dec.index[left]:
    dec.at[i, "final_pred"] = prior_pred(str(dec.at[i, "route"]), bool(dec.at[i, "has_ctx"]))
    dec.at[i, "final_layer"] = "PRIOR_safety"
print(f"safety-prior rows: {int(left.sum())}")

sub = dec[["id"]].copy()
sub["label"] = dec["final_pred"].astype(int)
sub = sub.sort_values("id").reset_index(drop=True)
assert sub["label"].isin([0, 1]).all() and len(sub) == len(dec)
sub.to_csv(WORK / "submission.csv", index=False)
dec.to_csv(WORK / "final_decisions_v3.csv", index=False)

print(f"\nFINAL layer census:\n{dec['final_layer'].value_counts().to_string()}")
print(f"\nprediction distribution: {sub['label'].value_counts().to_dict()}")
report = dict(model=MODEL_USED, workload=len(work2), qb_decided=n_qb,
              overrides=n_override, qbx_flips=flips,
              pred_dist={str(k): int(v) for k, v in sub["label"].value_counts().items()})
with open(WORK / "final_report_v3.json", "w") as f:
    _json.dump(report, f, indent=2)
print(_json.dumps(report, indent=2))
print("\nDONE — submit /kaggle/working/submission.csv")

workload overrides that changed a prediction: 585
cross-examining 98 disagree-rows against exam bank
QBX flips 0->1: 0 | QB-confirmed 0s: 29 | QB-vs-L2v2 conflicts on agree-rows: 1
safety-prior rows: 0

FINAL layer census:
final_layer
ENS_agree               526
L2v2_agree              524
QB2_tf                  487
LANG_ENS_agree          210
LANG_answer             153
L3_exact_substring      151
LLM_math                133
ENS_verdict             101
L1b_gold_disagree        98
L3_overlap_high          57
ENS_qb_tie               27
L3_digit_mismatch        18
L1a_exact_pair           11
LANG_QB_tie               6
L1b_wrong_agree           4
L1b_gold_agree            3
L3_relation_mismatch      3
PRIOR                     2
L3_type_mismatch          1
LANG_verdict              1

prediction distribution: {0: 1356, 1: 1160}
{
  "model": "Qwen/Qwen2.5-32B-Instruct-AWQ",
  "workload": 1646,
  "qb_decided": 487,
  "overrides": 585,
  "qbx_flips": 0,
  "pred_dist": {
    "0": 1356,
   

## Cell 11 — Snapshot

Zips all output artefacts into a single downloadable file for backup.

**Included files:** `submission.csv`, `final_decisions_v3.csv`, `final_report_v3.json`, embedding caches (`.npy`), BM25 index (`.pkl`), LLM generation cache (`llm_cache.csv`), and downloaded dataset archives.


In [11]:
import zipfile
from pathlib import Path
from datetime import datetime

def snapshot_to_zip(extra_patterns=None, zip_name=None, work_dir="/kaggle/working"):
    WORK = Path(work_dir)
    patterns = [
        "submission.csv", "final_decisions_v3.csv", "final_report_v3.json", "emb_*.npy",
        "layer_decisions.csv", "llm_cache.csv", "*.pkl",
        "brqa_*.json", "benqa.zip", "benqa_x/**/*.csv",
        "banglaquad.zip", "banglaquad_x/**/*.json",
    ]
    if extra_patterns: patterns += list(extra_patterns)
    files = sorted({f for pat in patterns for f in WORK.glob(pat) if f.is_file()})
    if not files: print("Nothing matched"); return None
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_name = zip_name or f"snapshot_{ts}.zip"
    zip_path = WORK / zip_name
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in files:
            zf.write(f, arcname=str(f.relative_to(WORK)))
    total_mb = sum(f.stat().st_size for f in files) / 1e6
    print(f"Zipped {len(files)} files ({total_mb:.1f} MB) -> {zip_path}")
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(zip_path.relative_to(WORK.parent)) if zip_path.is_relative_to(WORK.parent) else str(zip_path)))
    except Exception: pass
    return zip_path

snapshot_to_zip()

Zipped 42 files (1252.0 MB) -> /kaggle/working/snapshot_20260717_230909.zip


/kaggle/working/working/snapshot_20260717_230909.zip

PosixPath('/kaggle/working/snapshot_20260717_230909.zip')